<a href="https://colab.research.google.com/github/FBeni-04/MotoGP_prediction/blob/main/motogp_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MotoGP prediction

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, brier_score_loss, mean_absolute_error, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from IPython.display import HTML, display

from html import escape

try:
    from IPython.display import HTML, display
    HAS_IPYTHON = True
except ImportError:
    HAS_IPYTHON = False

warnings.filterwarnings("ignore")

INPUT = Path("motogp_dataset_with_junior_general_and_circuit_signals.csv")


PREDICTION_RACES = [
    {"round": 10, "circuit_id": "GER", "race_name": "German GP / Sachsenring"},
    {"round": 11, "circuit_id": "GBR", "race_name": "British GP / Silverstone"},
    {"round": 12, "circuit_id": "ARA", "race_name": "Aragon GP"},
    {"round": 13, "circuit_id": "RSM", "race_name": "San Marino GP / Misano"},
    {"round": 14, "circuit_id": "AUT", "race_name": "Austrian GP / Red Bull Ring"},
]
PREDICTION_YEAR = 2026
PREDICTION_ROUND = 10

REQUESTED_FULL_NAMES = [
    "Marco Bezzecchi",
    "Jorge Martin",
    "Raúl Fernández",
    "Ai Ogura",
    "Francesco Bagnaia",
    "Marc Márquez",
    "Fermín Aldeguer",
    "Alex Márquez",
    "Franco Morbidelli",
    "Fabio Di Giannantonio",
    #"Johann Zarco",
    "Diogo Moreira",
    "Luca Marini",
    "Joan Mir",
    "Brad Binder",
    "Pedro Acosta",
    "Maverick Vinales",
    "Enea Bastianini",
    "Fabio Quartararo",
    "Alex Rins",
    "Toprak Razgatlioglu",
    "Jack Miller",
    "Cal Crutchlow"
]

RIDER_IMAGE_URLS = {
    "Marco Bezzecchi": "https://resources.motogp.pulselive.com/photo-resources/2026/05/29/440d1ac6-83cd-4107-a831-efb8dd1eaa77/72-MGP-Marco-Bezzecchi-Rider-Official_DSC03346.png?width=1800",
    "Jorge Martin": "https://resources.motogp.pulselive.com/photo-resources/2026/05/29/56a55c22-98dd-4b18-85a2-f2919337776c/89-MGP-Jorge-Martin-Rider-Official_DSC03201.png?width=1800",
    "Raúl Fernández": "https://resources.motogp.pulselive.com/photo-resources/2026/06/19/e75830a0-7b4a-4ab0-9a57-6dad0241f298/dVSyS6Av.png?width=1800",
    "Ai Ogura": "https://resources.motogp.pulselive.com/photo-resources/2026/06/19/180a7c1a-fa1b-4cdf-ad00-184823d48b26/PxTIBU4p.png?width=1800",
    "Francesco Bagnaia": "https://resources.motogp.pulselive.com/photo-resources/2026/02/05/9772f542-8f9b-4a1c-b7a3-a5fe8f041f75/IfzOWPi2.png?width=1800",
    "Marc Márquez": "https://resources.motogp.pulselive.com/photo-resources/2026/04/17/e2312a70-2ff1-4b16-a3be-d1a3413879a5/Fvwnv4IB.png?width=1800",
    "Fermín Aldeguer": "https://resources.motogp.pulselive.com/photo-resources/2026/04/16/cef2a1da-7f3e-4530-9153-abc3076bb193/PzJyfoWK.png?width=1800",
    "Alex Márquez": "https://resources.motogp.pulselive.com/photo-resources/2026/02/05/71b70d16-3d66-4374-abf0-e439f76a13aa/WezEeZAR.png?width=1800",
    "Franco Morbidelli": "https://resources.motogp.pulselive.com/photo-resources/2026/04/16/665033a2-14e1-4eb2-b8b2-10f4b3a20345/Zz3uzT3q.png?width=1800",
    "Fabio Di Giannantonio": "https://resources.motogp.pulselive.com/photo-resources/2026/04/17/9fbe6dbc-bf82-4a39-ba02-0c6bbdacc81b/S4hTvWfV.png?width=1800",
    "Johann Zarco": "https://resources.motogp.pulselive.com/photo-resources/2026/02/05/49611a81-9931-4191-9820-068b73b54f99/y0R5f9H5.png?width=200",
    "Diogo Moreira": "https://resources.motogp.pulselive.com/photo-resources/2026/04/16/eebce0ce-4f64-4ef8-a723-f907d03ee651/bMBva6XJ.png?width=1800",
    "Luca Marini": "https://resources.motogp.pulselive.com/photo-resources/2026/04/16/90b11a99-50d5-4db4-a442-71e5b90b4852/XHS6Ttem.png?width=1800",
    "Joan Mir": "https://resources.motogp.pulselive.com/photo-resources/2026/04/16/2c2dbb56-b5eb-4230-a291-f0f65ce0fa4b/DabAuFuq.png?width=1800",
    "Brad Binder": "https://resources.motogp.pulselive.com/photo-resources/2026/05/15/bf875f3c-d9d0-4f8b-aa9a-124c7b9145b6/33-MGP-Brad-Binder-Rider-Official-x12-_DSC4264-1-.png?width=1800",
    "Pedro Acosta": "https://resources.motogp.pulselive.com/photo-resources/2026/05/14/5f2a2407-faa5-4887-a042-3d70b9861c57/MDd2dlRi.png?width=200",
    "Maverick Vinales": "https://resources.motogp.pulselive.com/photo-resources/2026/04/16/73414b18-23b7-426b-a6c4-b32d305c9aaf/LirohhSS.png?width=1800",
    "Enea Bastianini": "https://resources.motogp.pulselive.com/photo-resources/2026/02/05/32fd7aeb-d765-45d8-9da3-cc3ca25689cf/7pX3VTcG.png?width=1800",
    "Fabio Quartararo": "https://resources.motogp.pulselive.com/photo-resources/2026/02/05/73805511-aba7-4e37-9361-4e4b35da50fe/L72keLEc.png?width=1800",
    "Alex Rins": "https://resources.motogp.pulselive.com/photo-resources/2026/04/16/0ac2a783-b1da-4a1a-ba32-6e4ff5aa2e04/bTN2r4VS.png?width=1800",
    "Toprak Razgatlioglu": "https://resources.motogp.pulselive.com/photo-resources/2026/02/05/743b343d-2b20-40a7-8ae0-e4f5a273503d/5Zq5W4Wt.png?width=1800",
    "Jack Miller": "https://resources.motogp.pulselive.com/photo-resources/2026/06/05/85d57a3c-8997-4753-9801-a99f72fe9289/43-MGP-Jack-Miller-Rider-Official-x12-_DSC4139.png?width=1800",
    "Cal Crutchlow": "https://resources.motogp.pulselive.com/photo-resources/2023/09/29/697c26d7-fbdf-4d5e-a72e-6e3dc7f57429/_0001_35-Cal-Crutchlow_DS_4293.png?width=1800",
    "Augusto Fernández": "https://resources.motogp.pulselive.com/photo-resources/2026/04/24/2ecd63d5-6a4d-447a-9c02-d405be2d7636/47-MGP-Augusto-Fernandez-Rider-Official_DSC09642.png?width=1800",
}

RIDER_COUNTRY_FLAGS = {
    "Marco Bezzecchi": "🇮🇹",
    "Jorge Martin": "🇪🇸",
    "Raúl Fernández": "🇪🇸",
    "Ai Ogura": "🇯🇵",
    "Francesco Bagnaia": "🇮🇹",
    "Marc Márquez": "🇪🇸",
    "Fermín Aldeguer": "🇪🇸",
    "Alex Márquez": "🇪🇸",
    "Franco Morbidelli": "🇮🇹",
    "Fabio Di Giannantonio": "🇮🇹",
    "Johann Zarco": "🇫🇷",
    "Diogo Moreira": "🇧🇷",
    "Luca Marini": "🇮🇹",
    "Joan Mir": "🇪🇸",
    "Brad Binder": "🇿🇦",
    "Pedro Acosta": "🇪🇸",
    "Maverick Vinales": "🇪🇸",
    "Enea Bastianini": "🇮🇹",
    "Fabio Quartararo": "🇫🇷",
    "Alex Rins": "🇪🇸",
    "Toprak Razgatlioglu": "🇹🇷",
    "Jack Miller": "🇦🇺",
    "Cal Crutchlow": "🇬🇧",
    "Augusto Fernández": "🇪🇸",
}

TEAM_PROFILE_URLS = {
    "Aprilia Racing": "https://resources.motogp.pulselive.com/photo-resources/2026/05/29/1b9a59c0-1f0b-41e4-9ad5-3491ff572e0e/72-MGP-Marco-Bezzecchi-Bike_DSC03476.png?width=1248",
    "Ducati Lenovo Team": "https://resources.motogp.pulselive.com/photo-resources/2026/02/06/d1bd635d-955c-4d34-a571-b058309783ff/93-MGP-Marc-Marquez-Bike-Official-x01-LGZ_4375.png?width=1248",
    "Pertamina Enduro VR46 Racing Team": "https://resources.motogp.pulselive.com/photo-resources/2026/02/06/1e454664-72a0-4a50-8fdb-d67cb58ca919/Bike_MotoGP_2026-33.png?width=1248",
    "SuperFile Trackhouse MotoGP Team": "https://resources.motogp.pulselive.com/photo-resources/2026/06/19/24b758c2-8fff-4129-8a37-b3eddde86290/25-MGP-Raul-Fernandez-Bike-Official-x01-LGZ_3765.png?width=1248",
    "BK8 Gresini Racing MotoGP": "https://resources.motogp.pulselive.com/photo-resources/2026/02/06/4904a18b-9e65-49ca-b315-73ebde777e36/73-MGP-Alex-Marquez-Bike-Official-x01-LGZ_4297.png?width=1248",
    "Prima Pramac Yamaha MotoGP": "https://resources.motogp.pulselive.com/photo-resources/2026/02/06/ea882d51-ed5d-4c5e-a692-3bcb34a8b05a/07-MGP-Toprak-Razgatlioglu-Bike-Official-x01-LGZ_3962.png?width=1248",
    "Honda HRC Castrol": "https://resources.motogp.pulselive.com/photo-resources/2026/02/06/3cfcd3de-0bf9-466a-a39e-b2cb8f1a1f86/10-MGP-Luca-Marini-Bike-Official-x01-LGZ_3662.png?width=1248",
    "Monster Energy Yamaha MotoGP Team": "https://resources.motogp.pulselive.com/photo-resources/2026/02/06/1c5b6221-2788-4cd6-b39a-2d900fc1ca89/20-MGP-Fabio-Quartararo-Bike-Official-x01-LGZ_3593.png?width=1248",
    "Red Bull KTM Factory Racing": "https://resources.motogp.pulselive.com/photo-resources/2026/02/06/c9dbf5d1-6693-4d32-98a7-e34515675dcf/37-MGP-Pedro-Acosta-Bike-Official-x01-LGZ_4124.png?width=1248",
    "Castrol Honda LCR": "https://resources.motogp.pulselive.com/photo-resources/2026/02/09/7281bf2a-b61c-4677-b088-c296b6f493ee/05-MGP-Johann-Zarco-Bike-Official.png?width=1248",
    "Red Bull KTM Tech3": "https://resources.motogp.pulselive.com/photo-resources/2026/02/06/1b34fada-0f56-42b2-9713-3e26ec426959/12-MGP-Maverick-Vi-ales-Bike-Official-x01-LGZ_3866.png?width=1248",
    "Pro Honda LCR": "https://resources.motogp.pulselive.com/photo-resources/2026/02/09/7281bf2a-b61c-4677-b088-c296b6f493ee/05-MGP-Johann-Zarco-Bike-Official.png?width=1248"
}


JUNIOR_GENERAL_FEATURES = [
    "moto2_cc_weighted_avg_pos_signal",
    "moto2_cc_weighted_last10_avg_pos_signal",
    "moto2_general_model_influence_weight",

    "moto3_cc_weighted_avg_pos_signal",
    "moto3_cc_weighted_last10_avg_pos_signal",
    "moto3_general_model_influence_weight",
]

JUNIOR_CIRCUIT_FEATURES = [
    "moto2_same_circuit_cc_weighted_avg_pos_signal",
    "moto2_same_circuit_cc_weighted_last3_avg_pos_signal",
    "moto2_same_circuit_model_influence_weight",

    "moto3_same_circuit_cc_weighted_avg_pos_signal",
    "moto3_same_circuit_cc_weighted_last3_avg_pos_signal",
    "moto3_same_circuit_model_influence_weight",
]

JUNIOR_NUMERIC_FEATURES = JUNIOR_GENERAL_FEATURES + JUNIOR_CIRCUIT_FEATURES

JUNIOR_SCORE_CAP = 0.20


def img_tag(url: str, alt: str, width: int = 70) -> str:
    if not isinstance(url, str) or not url.startswith("http"):
        return ""

    return (
        f'<img src="{escape(url)}" '
        f'alt="{escape(alt)}" '
        f'width="{width}" '
        f'style="border-radius:10px; object-fit:cover;">'
    )


def make_prediction_table_html(
    all_pred: pd.DataFrame,
    rider_image_urls: dict,
    team_image_urls: dict,
) -> str:
    css = """
    <style>
        .motogp-wrap {
            font-family: Arial, sans-serif;
            max-width: 1500px;
        }

        .race-title {
            margin-top: 34px;
            margin-bottom: 10px;
            font-size: 24px;
            font-weight: 800;
        }

        table.motogp-table {
            border-collapse: collapse;
            width: 100%;
            margin-bottom: 28px;
            font-size: 14px;
        }

        table.motogp-table th {
            background: #111;
            color: white;
            padding: 9px;
            text-align: left;
        }

        table.motogp-table td {
            border-bottom: 1px solid #ddd;
            padding: 8px;
            vertical-align: middle;
        }

        .rank {
            font-size: 20px;
            font-weight: 800;
            text-align: center;
        }

        .rider-name {
            font-weight: 700;
            font-size: 15px;
        }

        .country {
            font-size: 24px;
            text-align: center;
        }

        .risk-alacsony {
            color: #087f23;
            font-weight: 700;
        }

        .risk-közepes {
            color: #b26a00;
            font-weight: 700;
        }

        .risk-magas {
            color: #b00020;
            font-weight: 700;
        }
    </style>
    """

    html = [css, '<div class="motogp-wrap">']

    for race_name, part in all_pred.groupby("race_name", sort=False):
        part = part.sort_values("predicted_rank").copy()

        html.append(f'<div class="race-title">{escape(str(race_name))}</div>')

        html.append(
            """
            <table class="motogp-table">
                <thead>
                    <tr>
                        <th>#</th>
                        <th>Versenyző képe</th>
                        <th>Versenyző</th>
                        <th>Nemzet</th>
                        <th>Csapat képe</th>
                        <th>Csapat</th>
                        <th>Várható score</th>
                        <th>Tempó score</th>
                        <th>Ha célba ér</th>
                        <th>Célba érés</th>
                        <th>DNF</th>
                        <th>Kockázat</th>
                    </tr>
                </thead>
                <tbody>
            """
        )

        for _, row in part.iterrows():
            rider = str(row["rider_full_name"])
            team = str(row["team"])

            rider_img = img_tag(
                rider_image_urls.get(rider, ""),
                rider,
                width=130,
            )

            team_img = img_tag(
                team_image_urls.get(team, ""),
                team,
                width=110,
            )

            country = RIDER_COUNTRY_FLAGS.get(rider, "")

            risk = str(row["risk_level"])
            risk_class = f"risk-{risk}"

            html.append(
                f"""
                <tr>
                    <td class="rank">{int(row["predicted_rank"])}</td>

                    <td>{rider_img}</td>

                    <td>
                        <div class="rider-name">{escape(rider)}</div>
                    </td>

                    <td class="country">{escape(country)}</td>

                    <td>{team_img}</td>

                    <td>{escape(team)}</td>

                    <td>{row["expected_position_score"]:.3f}</td>

                    <td>{row["pure_pace_score"]:.3f}</td>

                    <td>{row["predicted_position_if_finished"]:.3f}</td>

                    <td>{row["prob_finish"] * 100:.1f}%</td>

                    <td>{row["prob_dnf"] * 100:.1f}%</td>

                    <td class="{risk_class}">{escape(risk)}</td>
                </tr>
                """
            )

        html.append("</tbody></table>")

    html.append("</div>")

    return "\n".join(html)

def safe_one_hot_encoder() -> OneHotEncoder:
    """scikit-learn 1.2+ uses sparse_output, older versions use sparse."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def add_two_stage_features(data: pd.DataFrame) -> pd.DataFrame:
    """
    Feature engineering leakage nélkül.

    Fontos:
    - Ret / DNF / EX / NC nem 25. helyként megy bele a tempóátlagokba.
    - Ezek külön DNF / non-classified kockázatként szerepelnek.
    """
    data = data.sort_values(["year", "round", "circuit_id", "rider_id"]).copy()

    data["classified"] = (data["dnf"].fillna(0).astype(int) == 0).astype(int)
    data["finish_position_classified"] = np.where(
        data["classified"].eq(1),
        data["final_position"].astype(float),
        np.nan,
    )

    # Csak célba ért versenyek tempója, kiesések nélkül.
    for window in [3, 5, 10]:
        data[f"classified_avg_last{window}"] = data.groupby("rider_id")[
            "finish_position_classified"
        ].transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())

        data[f"dnf_rate_last{window}"] = data.groupby("rider_id")["dnf"].transform(
            lambda s: s.shift(1).rolling(window, min_periods=1).mean()
        )

        data[f"top10_rate_last{window}"] = data.groupby("rider_id")[
            "finish_position_classified"
        ].transform(lambda s: (s.shift(1) <= 10).rolling(window, min_periods=1).mean())

    # Szezonon belüli forma az adott futam előtt.
    data["season_classified_avg_before"] = data.groupby(["year", "rider_id"])[
        "finish_position_classified"
    ].transform(lambda s: s.shift(1).expanding().mean())

    data["season_dnf_rate_before"] = data.groupby(["year", "rider_id"])["dnf"].transform(
        lambda s: s.shift(1).expanding().mean()
    )

    data["season_points_before_calc"] = data.groupby(["year", "rider_id"])[
        "points"
    ].transform(lambda s: s.shift(1).fillna(0).cumsum())

    data["season_starts_before"] = data.groupby(["year", "rider_id"]).cumcount()

    # Teljes korábbi karrierforma az adott futam előtt.
    data["career_classified_avg_before"] = data.groupby("rider_id")[
        "finish_position_classified"
    ].transform(lambda s: s.shift(1).expanding().mean())

    data["career_dnf_rate_before"] = data.groupby("rider_id")["dnf"].transform(
        lambda s: s.shift(1).expanding().mean()
    )

    # Sprintforma.
    data["sprint_avg_last3"] = data.groupby("rider_id")["sprint_pos"].transform(
        lambda s: s.shift(1).rolling(3, min_periods=1).mean()
    )

    data["sprint_avg_last5"] = data.groupby("rider_id")["sprint_pos"].transform(
        lambda s: s.shift(1).rolling(5, min_periods=1).mean()
    )

    return data


def make_preprocessor(numeric_features, categorical_features):
    return ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), numeric_features),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", safe_one_hot_encoder()),
                    ]
                ),
                categorical_features,
            ),
        ],
        remainder="drop",
    )


def build_prediction_rows(
    df_feat: pd.DataFrame,
    circuit_id: str,
    prediction_round: int,
    race_name: str,
) -> pd.DataFrame:
    current = df_feat[df_feat["year"].eq(PREDICTION_YEAR)].copy()

    latest_by_rider = (
        current.sort_values(["round", "circuit_id"])
        .groupby("rider_full_name", as_index=False)
        .tail(1)
        .copy()
    )

    latest_by_rider = latest_by_rider[
        latest_by_rider["rider_full_name"].isin(REQUESTED_FULL_NAMES)
    ].copy()

    rows = []

    for _, last in latest_by_rider.iterrows():
        rider_id = last["rider_id"]
        constructor = last["constructor"]

        rider_hist = df_feat[df_feat["rider_id"].eq(rider_id)].sort_values(
            ["year", "round"]
        )

        season_hist = rider_hist[rider_hist["year"].eq(PREDICTION_YEAR)]
        classified_hist = rider_hist["finish_position_classified"].dropna()
        dnf_hist = rider_hist["dnf"].astype(float)
        sprint_hist = rider_hist["sprint_pos"].dropna()

        rider_circuit_hist = df_feat[
            df_feat["circuit_id"].eq(circuit_id)
            & df_feat["rider_id"].eq(rider_id)
        ].sort_values(["year", "round"])

        mfr_circuit_hist = df_feat[
            df_feat["circuit_id"].eq(circuit_id)
            & df_feat["constructor"].eq(constructor)
        ]

        rider_circuit_classified = rider_circuit_hist[
            "finish_position_classified"
        ].dropna()

        mfr_circuit_classified = mfr_circuit_hist[
            "finish_position_classified"
        ].dropna()

        row_data = {
            "year": PREDICTION_YEAR,
            "round": prediction_round,
            "circuit_id": circuit_id,
            "race_name": race_name,
            "rider_id": rider_id,
            "rider_name": last["rider_name"],
            "rider_full_name": last["rider_full_name"],
            "team": last["team"],
            "constructor": constructor,
            "motogp_experience_years": last.get("motogp_experience_years", np.nan),

            "classified_avg_last3": classified_hist.tail(3).mean(),
            "classified_avg_last5": classified_hist.tail(5).mean(),
            "classified_avg_last10": classified_hist.tail(10).mean(),
            "season_classified_avg_before": season_hist[
                "finish_position_classified"
            ].mean(),
            "career_classified_avg_before": classified_hist.mean(),
            "circuit_classified_avg": rider_circuit_classified.mean(),
            "mfr_circuit_classified_avg": mfr_circuit_classified.mean(),

            "dnf_rate_last3": dnf_hist.tail(3).mean(),
            "dnf_rate_last5": dnf_hist.tail(5).mean(),
            "dnf_rate_last10": dnf_hist.tail(10).mean(),
            "season_dnf_rate_before": season_hist["dnf"].mean(),
            "career_dnf_rate_before": dnf_hist.mean(),

            "top10_rate_last5": (classified_hist.tail(5) <= 10).mean(),
            "top10_rate_last10": (classified_hist.tail(10) <= 10).mean(),
            "sprint_avg_last3": sprint_hist.tail(3).mean(),
            "sprint_avg_last5": sprint_hist.tail(5).mean(),
            "season_points_before_calc": season_hist["points"].sum(),
            "season_starts_before": len(season_hist),
        }

        # Junior általános előélet:
        # mindig az utolsó aktuális 2026-os sorból vesszük.
        for col in JUNIOR_GENERAL_FEATURES:
            row_data[col] = last.get(col, np.nan)

        # Junior azonos pályás előélet:
        # az adott célpályára vonatkozó korábbi MotoGP-s sorból vesszük,
        # mert abban már elő van készítve a Moto2/Moto3 same-circuit jel.
        if not rider_circuit_hist.empty:
            latest_circuit_row = rider_circuit_hist.tail(1).iloc[0]
        else:
            latest_circuit_row = None

        for col in JUNIOR_CIRCUIT_FEATURES:
            if latest_circuit_row is not None:
                row_data[col] = latest_circuit_row.get(col, np.nan)
            else:
                row_data[col] = np.nan

        rows.append(row_data)

    pred_input = pd.DataFrame(rows)

    pred_input["champ_pos_before"] = pred_input[
        "season_points_before_calc"
    ].rank(method="min", ascending=False)

    return pred_input

def main():
    df = pd.read_csv(INPUT)

    df = df.sort_values(["year", "round", "circuit_id", "rider_id"]).reset_index(
        drop=True
    )

    df_feat = add_two_stage_features(df)

    numeric_features = [
        "round",
        "classified_avg_last3",
        "classified_avg_last5",
        "classified_avg_last10",
        "dnf_rate_last3",
        "dnf_rate_last5",
        "dnf_rate_last10",
        "top10_rate_last5",
        "top10_rate_last10",
        "season_classified_avg_before",
        "season_dnf_rate_before",
        "season_points_before_calc",
        "season_starts_before",
        "career_classified_avg_before",
        "career_dnf_rate_before",
        "sprint_avg_last3",
        "sprint_avg_last5",
        "motogp_experience_years",
        "champ_pos_before",
    ]

    numeric_features += JUNIOR_NUMERIC_FEATURES

    # Tanítási adatokhoz pálya- és gyártó-pálya statok.
    # Shift(1), hogy az adott futam eredménye ne szivárogjon be feature-ként.
    df_feat["circuit_classified_avg"] = df_feat.groupby(["circuit_id", "rider_id"])[
        "finish_position_classified"
    ].transform(lambda s: s.shift(1).expanding().mean())

    df_feat["mfr_circuit_classified_avg"] = df_feat.groupby(
        ["circuit_id", "constructor"]
    )["finish_position_classified"].transform(lambda s: s.shift(1).expanding().mean())

    numeric_features += [
        "circuit_classified_avg",
        "mfr_circuit_classified_avg",
    ]

    categorical_features = [
        "circuit_id",
        "rider_id",
        "constructor",
        "team",
    ]

    preprocessor = make_preprocessor(numeric_features, categorical_features)

    classifier_models = {
        "RF_cls": RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=4,
            class_weight="balanced_subsample",
            random_state=101,
            n_jobs=-1,
        ),
        "ET_cls": ExtraTreesClassifier(
            n_estimators=500,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=102,
            n_jobs=-1,
        ),
        "GB_cls": GradientBoostingClassifier(
            n_estimators=180,
            learning_rate=0.04,
            max_depth=3,
            random_state=103,
        ),
    }

    regressor_models = {
        "RF_reg": RandomForestRegressor(
            n_estimators=500,
            min_samples_leaf=4,
            random_state=201,
            n_jobs=-1,
        ),
        "ET_reg": ExtraTreesRegressor(
            n_estimators=500,
            min_samples_leaf=3,
            random_state=202,
            n_jobs=-1,
        ),
        "GB_reg": GradientBoostingRegressor(
            n_estimators=240,
            learning_rate=0.035,
            max_depth=3,
            subsample=0.85,
            random_state=203,
        ),
    }

    X_all = df_feat[numeric_features + categorical_features]
    y_finish = df_feat["classified"].astype(int)

    finishers_mask = df_feat["classified"].eq(1)

    y_pos_finishers = df_feat.loc[
        finishers_mask,
        "finish_position_classified",
    ].astype(float)

    report_rows = []

    # Validáció: 2022–2024 tanulás, 2025 teszt.
    train_mask = df_feat["year"].le(2024)
    val_mask = df_feat["year"].eq(2025)

    fitted_classifiers = {}

    for name, model in classifier_models.items():
        pipe = Pipeline(
            [
                ("prep", make_preprocessor(numeric_features, categorical_features)),
                ("model", model),
            ]
        )

        pipe.fit(X_all.loc[train_mask], y_finish.loc[train_mask])

        proba = pipe.predict_proba(X_all.loc[val_mask])[:, 1]
        pred_label = (proba >= 0.5).astype(int)
        y_true = y_finish.loc[val_mask]

        row = {
            "stage": "classifier_finish_probability",
            "model": name,
            "validation_year": 2025,
            "rows": int(val_mask.sum()),
            "accuracy": round(float(accuracy_score(y_true, pred_label)), 3),
            "brier_loss": round(float(brier_score_loss(y_true, proba)), 3),
        }

        try:
            row["roc_auc"] = round(float(roc_auc_score(y_true, proba)), 3)
        except ValueError:
            row["roc_auc"] = np.nan

        report_rows.append(row)

        final_pipe = Pipeline(
            [
                ("prep", make_preprocessor(numeric_features, categorical_features)),
                ("model", model),
            ]
        )

        final_pipe.fit(X_all, y_finish)
        fitted_classifiers[name] = final_pipe

    fitted_regressors = {}

    reg_train_mask = train_mask & finishers_mask
    reg_val_mask = val_mask & finishers_mask

    for name, model in regressor_models.items():
        pipe = Pipeline(
            [
                ("prep", make_preprocessor(numeric_features, categorical_features)),
                ("model", model),
            ]
        )

        pipe.fit(
            X_all.loc[reg_train_mask],
            df_feat.loc[reg_train_mask, "finish_position_classified"],
        )

        pred = np.clip(pipe.predict(X_all.loc[reg_val_mask]), 1, 24)

        mae = mean_absolute_error(
            df_feat.loc[reg_val_mask, "finish_position_classified"],
            pred,
        )

        report_rows.append(
            {
                "stage": "regressor_position_if_finished",
                "model": name,
                "validation_year": 2025,
                "rows": int(reg_val_mask.sum()),
                "mae_positions_only_finishers": round(float(mae), 3),
            }
        )

        final_pipe = Pipeline(
            [
                ("prep", make_preprocessor(numeric_features, categorical_features)),
                ("model", model),
            ]
        )

        final_pipe.fit(X_all.loc[finishers_mask], y_pos_finishers)
        fitted_regressors[name] = final_pipe

    all_predictions = []
    all_features = []

    for race in PREDICTION_RACES:
        pred_input = build_prediction_rows(
            df_feat=df_feat,
            circuit_id=race["circuit_id"],
            prediction_round=race["round"],
            race_name=race["race_name"],
        )

        present = set(pred_input["rider_full_name"].tolist())

        missing = [
            name
            for name in REQUESTED_FULL_NAMES
            if name not in present
        ]

        for missing_name in missing:
            report_rows.append(
                {
                    "stage": "prediction_input_warning",
                    "model": "missing_current_2026_row",
                    "race_name": race["race_name"],
                    "note": missing_name,
                }
            )

        for col in categorical_features:
            pred_input[col] = pred_input[col].astype(str)

        # 1. lépcső: célba érési valószínűség.
        for name, pipe in fitted_classifiers.items():
            pred_input[f"prob_finish_{name}"] = pipe.predict_proba(
                pred_input[numeric_features + categorical_features]
            )[:, 1]

        pred_input["prob_finish"] = (
            0.40 * pred_input["prob_finish_RF_cls"]
            + 0.35 * pred_input["prob_finish_ET_cls"]
            + 0.25 * pred_input["prob_finish_GB_cls"]
        )

        pred_input["prob_dnf"] = 1 - pred_input["prob_finish"]

        # 2. lépcső: várható helyezés, ha célba ér.
        for name, pipe in fitted_regressors.items():
            pred_input[f"pred_pos_{name}"] = np.clip(
                pipe.predict(pred_input[numeric_features + categorical_features]),
                1,
                24,
            )

        pred_input["predicted_position_if_finished"] = (
            0.40 * pred_input["pred_pos_RF_reg"]
            + 0.35 * pred_input["pred_pos_ET_reg"]
            + 0.25 * pred_input["pred_pos_GB_reg"]
        )

        fill_cols = [
            "season_classified_avg_before",
            "classified_avg_last5",
            "classified_avg_last3",
            "circuit_classified_avg",
            "mfr_circuit_classified_avg",
            "sprint_avg_last5",
        ]

        medians = pred_input[fill_cols].median(numeric_only=True)

        for col in fill_cols:
            pred_input[col + "_filled"] = pred_input[col].fillna(medians[col])

        pred_input["base_classified_form_score"] = (
            0.30 * pred_input["season_classified_avg_before_filled"]
            + 0.22 * pred_input["classified_avg_last5_filled"]
            + 0.13 * pred_input["classified_avg_last3_filled"]
            + 0.15 * pred_input["circuit_classified_avg_filled"]
            + 0.10 * pred_input["mfr_circuit_classified_avg_filled"]
            + 0.10 * pred_input["sprint_avg_last5_filled"]
        )

        junior_components = [
            (
                "moto2_cc_weighted_avg_pos_signal",
                "moto2_general_model_influence_weight",
            ),
            (
                "moto2_same_circuit_cc_weighted_avg_pos_signal",
                "moto2_same_circuit_model_influence_weight",
            ),
            (
                "moto3_cc_weighted_avg_pos_signal",
                "moto3_general_model_influence_weight",
            ),
            (
                "moto3_same_circuit_cc_weighted_avg_pos_signal",
                "moto3_same_circuit_model_influence_weight",
            ),
        ]

        pred_input["junior_raw_weight_sum"] = 0.0
        pred_input["junior_weighted_score_sum"] = 0.0

        for signal_col, weight_col in junior_components:
            if signal_col not in pred_input.columns:
                pred_input[signal_col] = np.nan

            if weight_col not in pred_input.columns:
                pred_input[weight_col] = np.nan

            valid = pred_input[signal_col].notna() & pred_input[weight_col].notna()

            pred_input.loc[valid, "junior_raw_weight_sum"] += pred_input.loc[
                valid, weight_col
            ]

            pred_input.loc[valid, "junior_weighted_score_sum"] += (
                pred_input.loc[valid, signal_col]
                * pred_input.loc[valid, weight_col]
            )

        # Maximum 20% juniorhatás.
        pred_input["junior_effective_weight_sum"] = pred_input[
            "junior_raw_weight_sum"
        ].clip(upper=JUNIOR_SCORE_CAP)

        pred_input["junior_weight_scale"] = np.where(
            pred_input["junior_raw_weight_sum"] > 0,
            pred_input["junior_effective_weight_sum"]
            / pred_input["junior_raw_weight_sum"],
            0.0,
        )

        pred_input["junior_score_component"] = (
            pred_input["junior_weighted_score_sum"]
            * pred_input["junior_weight_scale"]
        )

        pred_input["base_score_weight"] = 1.0 - pred_input["junior_effective_weight_sum"]

        pred_input["classified_form_score"] = (
            pred_input["base_score_weight"] * pred_input["base_classified_form_score"]
            + pred_input["junior_score_component"]
        )

        pred_input["pure_pace_score"] = (
            0.60 * pred_input["predicted_position_if_finished"]
            + 0.40 * pred_input["classified_form_score"]
        )

        pred_input["dnf_risk_penalty"] = (
            4.0 * pred_input["prob_dnf"]
            + 1.2 * pred_input["season_dnf_rate_before"].fillna(0)
            + 0.8 * pred_input["dnf_rate_last5"].fillna(0)
        )

        pred_input["expected_position_score"] = (
            pred_input["pure_pace_score"] + pred_input["dnf_risk_penalty"]
        )

        pred_input["risk_level"] = pd.cut(
            pred_input["prob_dnf"],
            bins=[-0.01, 0.25, 0.45, 1.0],
            labels=["alacsony", "közepes", "magas"],
        )

        pred_input = pred_input.sort_values("expected_position_score").reset_index(
            drop=True
        )

        pred_input["predicted_rank"] = np.arange(1, len(pred_input) + 1)

        pred_input["pure_pace_rank"] = pred_input["pure_pace_score"].rank(
            method="first",
            ascending=True,
        )

        pred_input["expected_rank"] = pred_input["expected_position_score"].rank(
            method="first",
            ascending=True,
        )

        all_predictions.append(pred_input.copy())
        all_features.append(pred_input.copy())

    all_pred = pd.concat(all_predictions, ignore_index=True)

    out_cols = [
        "race_name",
        "circuit_id",
        "round",
        "predicted_rank",
        "rider_full_name",
        "constructor",
        "team",
        "expected_position_score",
        "pure_pace_score",
        "predicted_position_if_finished",
        "prob_finish",
        "prob_dnf",
        "dnf_risk_penalty",
        "risk_level",

        "base_classified_form_score",
        "classified_form_score",
        "junior_effective_weight_sum",
        "junior_score_component",

        "moto2_cc_weighted_avg_pos_signal",
        "moto2_same_circuit_cc_weighted_avg_pos_signal",
        "moto3_cc_weighted_avg_pos_signal",
        "moto3_same_circuit_cc_weighted_avg_pos_signal",

        "season_points_before_calc",
        "champ_pos_before",
        "season_classified_avg_before",
        "classified_avg_last3",
        "classified_avg_last5",
        "classified_avg_last10",
        "circuit_classified_avg",
        "mfr_circuit_classified_avg",
        "sprint_avg_last5",
    ]

    all_pred = all_pred[out_cols].copy()

    html = make_prediction_table_html(
        all_pred=all_pred,
        rider_image_urls=RIDER_IMAGE_URLS,
        team_image_urls=TEAM_PROFILE_URLS,
    )
    HTML_OUT = Path("motogp_next5_predictions.html")
    HTML_OUT.write_text(html, encoding="utf-8")
    if HAS_IPYTHON:
        display(HTML(html))
    else:
        print(html)

    print("\nValidáció:")
    print(pd.DataFrame(report_rows).to_string(index=False))


if __name__ == "__main__":
    main()

#,Versenyző képe,Versenyző,Nemzet,Csapat képe,Csapat,Várható score,Tempó score,Ha célba ér,Célba érés,DNF,Kockázat
1,,Jorge Martin,🇪🇸,,Aprilia Racing,5.862,3.814,3.350,62.8%,37.2%,közepes
2,,Fabio Di Giannantonio,🇮🇹,,Pertamina Enduro VR46 Racing Team,6.045,4.749,4.242,67.6%,32.4%,közepes
3,,Marco Bezzecchi,🇮🇹,,Aprilia Racing,6.225,3.838,3.718,61.3%,38.7%,közepes
4,,Marc Márquez,🇪🇸,,Ducati Lenovo Team,6.503,5.036,4.815,70.0%,30.0%,közepes
5,,Francesco Bagnaia,🇮🇹,,Ducati Lenovo Team,6.721,4.236,4.198,53.9%,46.1%,magas
6,,Ai Ogura,🇯🇵,,SuperFile Trackhouse MotoGP Team,7.544,5.997,5.180,64.3%,35.7%,közepes
7,,Raúl Fernández,🇪🇸,,SuperFile Trackhouse MotoGP Team,9.025,7.503,7.039,69.0%,31.0%,közepes
8,,Alex Márquez,🇪🇸,,BK8 Gresini Racing MotoGP,9.158,7.238,7.113,71.2%,28.8%,közepes
9,,Pedro Acosta,🇪🇸,,Red Bull KTM Factory Racing,9.411,6.749,7.185,54.5%,45.5%,magas
10,,Luca Marini,🇮🇹,,Honda HRC Castrol,9.832,9.098,8.954,81.7%,18.3%,alacsony



Validáció:
                         stage  model  validation_year  rows  accuracy  brier_loss  roc_auc  mae_positions_only_finishers
 classifier_finish_probability RF_cls             2025   410     0.793       0.176    0.587                           NaN
 classifier_finish_probability ET_cls             2025   410     0.746       0.189    0.595                           NaN
 classifier_finish_probability GB_cls             2025   410     0.780       0.167    0.599                           NaN
regressor_position_if_finished RF_reg             2025   323       NaN         NaN      NaN                         3.212
regressor_position_if_finished ET_reg             2025   323       NaN         NaN      NaN                         3.358
regressor_position_if_finished GB_reg             2025   323       NaN         NaN      NaN                         3.306


# Moto2 Prediction

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, brier_score_loss, mean_absolute_error, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from IPython.display import HTML, display

from html import escape

try:
    from IPython.display import HTML, display
    HAS_IPYTHON = True
except ImportError:
    HAS_IPYTHON = False

warnings.filterwarnings("ignore")

INPUT = Path("moto2_dataset_from_Race2_plus_2026_current.csv")


PREDICTION_RACES = [
    {"round": 10, "circuit_id": "GER", "race_name": "German GP / Sachsenring"},
    {"round": 11, "circuit_id": "GBR", "race_name": "British GP / Silverstone"},
    {"round": 12, "circuit_id": "ARA", "race_name": "Aragon GP"},
    {"round": 13, "circuit_id": "RSM", "race_name": "San Marino GP / Misano"},
    {"round": 14, "circuit_id": "AUT", "race_name": "Austrian GP / Red Bull Ring"},
]
PREDICTION_YEAR = 2026
PREDICTION_ROUND = 10

REQUESTED_FULL_NAMES = [
    "Manuel González",
    "Izan Guevara",
    "Senna Agius",
    "David Alonso",
    "Celestino Vietti",
    "Daniel Holgado",
    "Iván Ortolá",
    "Filip Salac",
    "Alonso López",
    "Collin Veijer",
    "Daniel Muñoz",
    "Álex Escrig",
    "Tony Arbolino",
    "Barry Baltus",
    "Joe Roberts",
    "Adrián Huertas",
    "José Antonio Rueda",
    "Deniz Öncü",
    "Alberto Fernández",
    "Arón Canet",
    "Zonta van den Goorbergh",
    "Luca Lunetta",
    "Ayumu Sasaki",
    "Sergio García",
    "Taiyo Furusato",
    #"Mario Aji",
    "Ángel Piqueras",
    "Jorge Navarro",
    "Xabi Zurutuza",
    "Jacob Roulstone",
    "Milan Pawelec",
]

RIDER_IMAGE_URLS = {
    "Manuel González": "https://resources.motogp.pulselive.com/photo-resources/2026/04/16/551b6034-bef9-4c50-a764-da77b521313c/hnW1tygL.png?width=1800",
    "Izan Guevara": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/11ea27bf-56c5-40e5-82c4-b4f2f8cd4e89/Cz1duhOC.png?width=1800",
    "Senna Agius": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/e2a75782-d9f7-4b65-bb8a-fef5f8c7fb8a/yfGdM3eM.png?width=1800",
    "David Alonso": "https://resources.motogp.pulselive.com/photo-resources/2026/06/26/1daf00f6-e04c-4662-923b-c6f901011d1c/GBjWkovE.png?width=1800",
    "Celestino Vietti": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/5cb6378d-137e-4ee3-8632-760f271d40d8/9NzY6VrP.png?width=1800",
    "Daniel Holgado": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/243e5f51-9be3-4e13-a30b-a40daebb4614/Im46l63J.png?width=1800",
    "Iván Ortolá": "https://resources.motogp.pulselive.com/photo-resources/2026/06/25/c560dde2-69d0-418f-846d-48f53e639fda/bRNc5r4d.png?width=1800",
    "Filip Salac": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/e46f2dde-2c41-4330-b9df-b4525d2ce175/H8bjpsjN.png?width=1800",
    "Alonso López": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/2cd1efe5-13f7-4636-bda4-a1f3461d1c35/TmEYOgMO.png?width=1800",
    "Collin Veijer": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/17b2f118-fdf7-4e92-ad15-7792d7c4e966/s6YBio5b.png?width=1800",
    "Daniel Muñoz": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/07a01f1e-c18d-4860-a737-608b94ea24ad/bbVfeGjv.png?width=1800",
    "Álex Escrig": "https://resources.motogp.pulselive.com/photo-resources/2024/03/05/08756729-c9f6-4d50-bb41-4f1cbb71cfc0/11_Alex_Escrig_Moto2_Rider_DS_2368.png?width=1800",
    "Tony Arbolino": "https://resources.motogp.pulselive.com/photo-resources/2026/03/28/2cfb40b4-76f4-41ea-9e87-d5a0b9813166/2i91C2qg.png?width=1800",
    "Barry Baltus": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/bb2b307d-e16b-4cdc-a77b-68317966755b/9IOqHICT.png?width=1800",
    "Joe Roberts": "https://resources.motogp.pulselive.com/photo-resources/2024/03/07/b404a2fe-fc43-48f0-b0cf-779f1a0ebfef/16-Joe-Roberts-Moto2-Rider-DS-2564.png?width=1800",
    "Adrián Huertas": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/74a77da8-8d68-4219-9fcd-2b73c778f8d2/7LkOFkCA.png?width=1800",
    "José Antonio Rueda": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/993a3b65-6b1f-4edf-b8b9-2402af625997/J7a8ZI7x.png?width=1800",
    "Deniz Öncü": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/8d562fa1-309e-456e-999c-ce75e44db638/tQylx0zg.png?width=1800",
    "Alberto Fernández": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/768a7112-2fde-4b27-8dc9-2be5b24c49ba/NlxOasE2.png?width=1800",
    "Arón Canet": "https://resources.motogp.pulselive.com/photo-resources/2026/04/16/f717bbc4-aa01-445d-84bc-3b7d36123f34/90jDSvmV.png?width=1800",
    "Zonta van den Goorbergh": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/23d13547-c006-4cff-b109-58290dc963d3/0I6jkDPL.png?width=1800",
    "Luca Lunetta": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/42fa87f3-1ed4-4573-bfaa-c2a99b803f9f/NdoJK2jK.png?width=1800",
    "Ayumu Sasaki": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/042e9529-4bdb-4fac-a02e-7c7baf650a48/PqMe5d3Z.png?width=1800",
    "Sergio García": "https://resources.motogp.pulselive.com/photo-resources/2025/11/07/3098c097-ebe6-438c-8615-673b8a8f5ff8/KVM5xr1H.png?width=1800",
    "Taiyo Furusato": "https://resources.motogp.pulselive.com/photo-resources/2026/04/16/67548079-c593-4c05-a183-e787a86ba289/ib0oYDhI.png?width=1800",
    #"Mario Aji": "https://resources.motogp.pulselive.com/photo-resources/2025/02/24/2b3791b3-f806-4e85-b8a1-b3b7227bd096/4tlbaB0L.png?width=1800",
    "Ángel Piqueras": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/d041c55f-04e7-4efc-8c8a-97745000538e/1jSZ8Z6p.png?width=1800",
    "Jorge Navarro": "https://resources.motogp.pulselive.com/photo-resources/2026/06/26/cebfbec0-0dd1-44fe-b238-5e576f16bc21/09-M2-Jorge-Navarro-Rider-OfficialLGZ_3886.png?width=1800",
    "Xabi Zurutuza": "https://resources.motogp.pulselive.com/photo-resources/2026/05/14/7802bb80-d172-4a29-87f6-f79e9d8a2586/En2DcwAO.png?width=1800",
    "Jacob Roulstone": "https://resources.motogp.pulselive.com/photo-resources/2026/06/05/f3040636-9c65-477a-a2f4-e472f4cdd97e/ZL7NEPie.png?width=1800",
    "Milan Pawelec": "https://resources.motogp.pulselive.com/photo-resources/2026/06/26/6985fccb-8a86-449b-a21a-ceb1ec4d4b5b/40-M2-Milan-Pawelec-Rider-OfficialLGZ_3817.png?width=1800",
}

RIDER_COUNTRY_FLAGS = {
    "Manuel González": "🇪🇸",
    "Izan Guevara": "🇪🇸",
    "Senna Agius": "🇦🇺",
    "David Alonso": "🇨🇴",
    "Celestino Vietti": "🇮🇹",
    "Daniel Holgado": "🇪🇸",
    "Iván Ortolá": "🇪🇸",
    "Filip Salac": "🇨🇿",
    "Alonso López": "🇪🇸",
    "Collin Veijer": "🇳🇱",
    "Daniel Muñoz": "🇪🇸",
    "Álex Escrig": "🇪🇸",
    "Tony Arbolino": "🇮🇹",
    "Barry Baltus": "🇧🇪",
    "Joe Roberts": "🇺🇸",
    "Adrián Huertas": "🇪🇸",
    "José Antonio Rueda": "🇪🇸",
    "Deniz Öncü": "🇹🇷",
    "Alberto Fernández": "🇪🇸",
    "Arón Canet": "🇪🇸",
    "Zonta van den Goorbergh": "🇳🇱",
    "Luca Lunetta": "🇮🇹",
    "Ayumu Sasaki": "🇯🇵",
    "Sergio García": "🇪🇸",
    "Taiyo Furusato": "🇯🇵",
    "Mario Aji": "🇮🇩",
    "Ángel Piqueras": "🇪🇸",
    "Marcos Ramírez": "🇪🇸",
    "Unai Orradre": "🇪🇸",
    "Jorge Navarro": "🇪🇸",
    "Dennis Foggia": "🇮🇹",
    "Xabi Zurutuza": "🇪🇸",
    "Jacob Roulstone": "🇦🇺",
    "Milan Pawelec": "🇵🇱",
}

TEAM_PROFILE_URLS = {
    "Liqui Moly Dynavolt Intact GP": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/86b1330c-33a1-4c43-8955-c0c9194c8387/18-M2-Manuel-Gonzalez-Bike-Official-x01-LG9_0979.png?width=1248",
    "Blu Cru Pramac Yamaha Moto2": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/205735e8-2aae-48d1-aa34-25466c453550/28-M2-Izan-Guevara-Bike-Official-x01-LG9_0394.png?width=1248",
    "CFMoto Aspar Team": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/7a6887cf-cb42-4cb1-883a-6484b56514c2/80-M2-David-Alonso-Bike-Official-x01-LG9_0619.png?width=1248",
    "SpeedRS Team": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/fbbbcbe7-610e-4bdf-bd5d-9895667b3594/13-M2-Celestino-Vietti-Bike-Official-x01-LG9_1001.png?width=1248",
    "QJMotor – MSi": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/da42b0cb-3e2f-4f2c-b64d-4fde36b7e9a7/04-M2-Ivan-Ortola-Bike-Official-x01-LG9_1074.png?width=1248",
    "OnlyFans American Racing Team": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/ea911e04-65a4-4a7e-b9f1-7e97d7a8a114/12-M2-Filip-Salac-Bike-Official-x01-LG9_0650.png?width=1248",
    "Italjet Gresini Moto2": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/a54a7a6a-8a66-4cf3-87cd-d293998013a9/21-M2-Alonso-Lopez-Bike-Official-x01-LG9_0908.png?width=1248",
    "Red Bull KTM Ajo": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/2ace7c17-4beb-455d-8f26-37fa4e883077/95-M2-Collin-Veijer-Bike-Official-x01-LG9_0825.png?width=1248",
    "Italtrans Racing Team": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/c7199e5d-0774-48a1-aaea-eae2a8f06bf2/99-M2-Adrian-Huertas-Bike-Official-x01-LG9_0493.png?width=1248",
    "Klint Racing Team": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/350f6dbd-6bd1-493c-8875-a303c0b2393d/09-M2-Jorge-Navarro-Bike-Official-x01-LG9_0147.png?width=1248",
    "Reds Fantic Racing": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/a5f84835-572e-49ef-aff8-1b61d78ef6d4/07-M2-Barry-Baltus-Bike-Official-x01-LG9_0781.png?width=1248",
    "Elf Marc VDS Racing Team": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/5ee180d4-0516-4598-bf40-05aa7c1a93f6/44-M2-Aron-Canet-Bike-Official-x01-LG9_0735.png?width=1248",
    "Momoven Idrofoglia RW Racing Team": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/2ccb4e87-95df-44be-a707-75e3fcc8ac91/71-M2-Ayumu-Sasaki-Bike-Official-x01-LG9_9718.png?width=1248",
    "Idemitsu Honda Team Asia": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/b5d20705-a527-43ed-8c63-4401e560cc2c/72-M2-Taiyo-Furusato-Bike-Official-x01-LG9_0516.png?width=1248",
}

def img_tag(url: str, alt: str, width: int = 70) -> str:
    if not isinstance(url, str) or not url.startswith("http"):
        return ""

    return (
        f''
    )


def make_prediction_table_html(
    all_pred: pd.DataFrame,
    rider_image_urls: dict,
    team_image_urls: dict,
) -> str:
    css = """

    """

    html = [css, '']

    for race_name, part in all_pred.groupby("race_name", sort=False):
        part = part.sort_values("predicted_rank").copy()

        html.append(f'{escape(str(race_name))}')

        html.append(
            """

            """
        )

        for _, row in part.iterrows():
            rider = str(row["rider_full_name"])
            team = str(row["team"])

            rider_img = img_tag(
                rider_image_urls.get(rider, ""),
                rider,
                width=130,
            )

            team_img = img_tag(
                team_image_urls.get(team, ""),
                team,
                width=110,
            )

            country = RIDER_COUNTRY_FLAGS.get(rider, "")

            risk = str(row["risk_level"])
            risk_class = f"risk-{risk}"

            html.append(
                f"""

                """
            )

        html.append("


                        #
                        Versenyző képe
                        Versenyző
                        Nemzet
                        Csapat képe
                        Csapat
                        Várható score
                        Tempó score
                        Ha célba ér
                        Célba érés
                        DNF
                        Kockázat



                    {int(row["predicted_rank"])}

                    {rider_img}


                        {escape(rider)}


                    {escape(country)}

                    {team_img}

                    {escape(team)}

                    {row["expected_position_score"]:.3f}

                    {row["pure_pace_score"]:.3f}

                    {row["predicted_position_if_finished"]:.3f}

                    {row["prob_finish"] * 100:.1f}%

                    {row["prob_dnf"] * 100:.1f}%

                    {escape(risk)}
                ")

    html.append("")

    return "\n".join(html)

def safe_one_hot_encoder() -> OneHotEncoder:
    """scikit-learn 1.2+ uses sparse_output, older versions use sparse."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def add_two_stage_features(data: pd.DataFrame) -> pd.DataFrame:
    """
    Feature engineering leakage nélkül.

    Fontos:
    - Ret / DNF / EX / NC nem 25. helyként megy bele a tempóátlagokba.
    - Ezek külön DNF / non-classified kockázatként szerepelnek.
    """
    data = data.sort_values(["year", "round", "circuit_id", "rider_id"]).copy()

    data["classified"] = (data["dnf"].fillna(0).astype(int) == 0).astype(int)
    data["finish_position_classified"] = np.where(
        data["classified"].eq(1),
        data["final_position"].astype(float),
        np.nan,
    )

    # Csak célba ért versenyek tempója, kiesések nélkül.
    for window in [3, 5, 10]:
        data[f"classified_avg_last{window}"] = data.groupby("rider_id")[
            "finish_position_classified"
        ].transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())

        data[f"dnf_rate_last{window}"] = data.groupby("rider_id")["dnf"].transform(
            lambda s: s.shift(1).rolling(window, min_periods=1).mean()
        )

        data[f"top10_rate_last{window}"] = data.groupby("rider_id")[
            "finish_position_classified"
        ].transform(lambda s: (s.shift(1) <= 10).rolling(window, min_periods=1).mean())

    # Szezonon belüli forma az adott futam előtt.
    data["season_classified_avg_before"] = data.groupby(["year", "rider_id"])[
        "finish_position_classified"
    ].transform(lambda s: s.shift(1).expanding().mean())

    data["season_dnf_rate_before"] = data.groupby(["year", "rider_id"])["dnf"].transform(
        lambda s: s.shift(1).expanding().mean()
    )

    data["season_points_before_calc"] = data.groupby(["year", "rider_id"])[
        "points"
    ].transform(lambda s: s.shift(1).fillna(0).cumsum())

    data["season_starts_before"] = data.groupby(["year", "rider_id"]).cumcount()

    # Teljes korábbi karrierforma az adott futam előtt.
    data["career_classified_avg_before"] = data.groupby("rider_id")[
        "finish_position_classified"
    ].transform(lambda s: s.shift(1).expanding().mean())

    data["career_dnf_rate_before"] = data.groupby("rider_id")["dnf"].transform(
        lambda s: s.shift(1).expanding().mean()
    )

    return data


def make_preprocessor(numeric_features, categorical_features):
    return ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), numeric_features),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", safe_one_hot_encoder()),
                    ]
                ),
                categorical_features,
            ),
        ],
        remainder="drop",
    )


def build_prediction_rows(df_feat: pd.DataFrame, circuit_id: str, prediction_round: int, race_name: str) -> pd.DataFrame:
    current = df_feat[df_feat["year"].eq(PREDICTION_YEAR)].copy()

    latest_by_rider = (
        current.sort_values(["round", "circuit_id"])
        .groupby("rider_full_name", as_index=False)
        .tail(1)
        .copy()
    )

    latest_by_rider = latest_by_rider[
        latest_by_rider["rider_full_name"].isin(REQUESTED_FULL_NAMES)
    ].copy()

    rows = []

    for _, last in latest_by_rider.iterrows():
        rider_id = last["rider_id"]
        constructor = last["bike"]

        rider_hist = df_feat[df_feat["rider_id"].eq(rider_id)].sort_values(
            ["year", "round"]
        )

        season_hist = rider_hist[rider_hist["year"].eq(PREDICTION_YEAR)]
        classified_hist = rider_hist["finish_position_classified"].dropna()
        dnf_hist = rider_hist["dnf"].astype(float)

        rider_circuit_hist = df_feat[
            df_feat["circuit_id"].eq(circuit_id)
            & df_feat["rider_id"].eq(rider_id)
        ]

        mfr_circuit_hist = df_feat[
            df_feat["circuit_id"].eq(circuit_id)
            & df_feat["bike"].eq(constructor)
        ]

        rider_circuit_classified = rider_circuit_hist[
            "finish_position_classified"
        ].dropna()

        mfr_circuit_classified = mfr_circuit_hist[
            "finish_position_classified"
        ].dropna()

        rows.append(
            {
                "year": PREDICTION_YEAR,
                "round": prediction_round,
                "circuit_id": circuit_id,
                "race_name": race_name,
                "rider_id": rider_id,
                "rider_name": last["rider_name"],
                "rider_full_name": last["rider_full_name"],
                "team": last["team"],
                "bike": constructor,
                "moto2_experience_years": last.get("moto2_experience_years", np.nan),

                "classified_avg_last3": classified_hist.tail(3).mean(),
                "classified_avg_last5": classified_hist.tail(5).mean(),
                "classified_avg_last10": classified_hist.tail(10).mean(),
                "season_classified_avg_before": season_hist[
                    "finish_position_classified"
                ].mean(),
                "career_classified_avg_before": classified_hist.mean(),
                "circuit_classified_avg": rider_circuit_classified.mean(),
                "mfr_circuit_classified_avg": mfr_circuit_classified.mean(),

                "dnf_rate_last3": dnf_hist.tail(3).mean(),
                "dnf_rate_last5": dnf_hist.tail(5).mean(),
                "dnf_rate_last10": dnf_hist.tail(10).mean(),
                "season_dnf_rate_before": season_hist["dnf"].mean(),
                "career_dnf_rate_before": dnf_hist.mean(),

                "top10_rate_last5": (classified_hist.tail(5) <= 10).mean(),
                "top10_rate_last10": (classified_hist.tail(10) <= 10).mean(),
                "season_points_before_calc": season_hist["points"].sum(),
                "season_starts_before": len(season_hist),
            }
        )

    pred_input = pd.DataFrame(rows)

    pred_input["champ_pos_before"] = pred_input[
        "season_points_before_calc"
    ].rank(method="min", ascending=False)

    return pred_input

def main():
    df = pd.read_csv(INPUT)

    df = df.sort_values(["year", "round", "circuit_id", "rider_id"]).reset_index(
        drop=True
    )

    df_feat = add_two_stage_features(df)

    numeric_features = [
        "round",
        "classified_avg_last3",
        "classified_avg_last5",
        "classified_avg_last10",
        "dnf_rate_last3",
        "dnf_rate_last5",
        "dnf_rate_last10",
        "top10_rate_last5",
        "top10_rate_last10",
        "season_classified_avg_before",
        "season_dnf_rate_before",
        "season_points_before_calc",
        "season_starts_before",
        "career_classified_avg_before",
        "career_dnf_rate_before",
        "moto2_experience_years",
        "champ_pos_before",
    ]

    # Tanítási adatokhoz pálya- és gyártó-pálya statok.
    # Shift(1), hogy az adott futam eredménye ne szivárogjon be feature-ként.
    df_feat["circuit_classified_avg"] = df_feat.groupby(["circuit_id", "rider_id"])[
        "finish_position_classified"
    ].transform(lambda s: s.shift(1).expanding().mean())

    df_feat["mfr_circuit_classified_avg"] = df_feat.groupby(
        ["circuit_id", "bike"]
    )["finish_position_classified"].transform(lambda s: s.shift(1).expanding().mean())

    numeric_features += [
        "circuit_classified_avg",
        "mfr_circuit_classified_avg",
    ]

    categorical_features = [
        "circuit_id",
        "rider_id",
        "bike",
        "team",
    ]

    preprocessor = make_preprocessor(numeric_features, categorical_features)

    classifier_models = {
        "RF_cls": RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=4,
            class_weight="balanced_subsample",
            random_state=101,
            n_jobs=-1,
        ),
        "ET_cls": ExtraTreesClassifier(
            n_estimators=500,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=102,
            n_jobs=-1,
        ),
        "GB_cls": GradientBoostingClassifier(
            n_estimators=180,
            learning_rate=0.04,
            max_depth=3,
            random_state=103,
        ),
    }

    regressor_models = {
        "RF_reg": RandomForestRegressor(
            n_estimators=500,
            min_samples_leaf=4,
            random_state=201,
            n_jobs=-1,
        ),
        "ET_reg": ExtraTreesRegressor(
            n_estimators=500,
            min_samples_leaf=3,
            random_state=202,
            n_jobs=-1,
        ),
        "GB_reg": GradientBoostingRegressor(
            n_estimators=240,
            learning_rate=0.035,
            max_depth=3,
            subsample=0.85,
            random_state=203,
        ),
    }

    X_all = df_feat[numeric_features + categorical_features]
    y_finish = df_feat["classified"].astype(int)

    finishers_mask = df_feat["classified"].eq(1)

    y_pos_finishers = df_feat.loc[
        finishers_mask,
        "finish_position_classified",
    ].astype(float)

    report_rows = []

    # Validáció: 2022–2024 tanulás, 2025 teszt.
    train_mask = df_feat["year"].le(2024)
    val_mask = df_feat["year"].eq(2025)

    fitted_classifiers = {}

    for name, model in classifier_models.items():
        pipe = Pipeline(
            [
                ("prep", make_preprocessor(numeric_features, categorical_features)),
                ("model", model),
            ]
        )

        pipe.fit(X_all.loc[train_mask], y_finish.loc[train_mask])

        proba = pipe.predict_proba(X_all.loc[val_mask])[:, 1]
        pred_label = (proba >= 0.5).astype(int)
        y_true = y_finish.loc[val_mask]

        row = {
            "stage": "classifier_finish_probability",
            "model": name,
            "validation_year": 2025,
            "rows": int(val_mask.sum()),
            "accuracy": round(float(accuracy_score(y_true, pred_label)), 3),
            "brier_loss": round(float(brier_score_loss(y_true, proba)), 3),
        }

        try:
            row["roc_auc"] = round(float(roc_auc_score(y_true, proba)), 3)
        except ValueError:
            row["roc_auc"] = np.nan

        report_rows.append(row)

        final_pipe = Pipeline(
            [
                ("prep", preprocessor),
                ("model", model),
            ]
        )

        final_pipe.fit(X_all, y_finish)
        fitted_classifiers[name] = final_pipe

    fitted_regressors = {}

    reg_train_mask = train_mask & finishers_mask
    reg_val_mask = val_mask & finishers_mask

    for name, model in regressor_models.items():
        pipe = Pipeline(
            [
                ("prep", make_preprocessor(numeric_features, categorical_features)),
                ("model", model),
            ]
        )

        pipe.fit(
            X_all.loc[reg_train_mask],
            df_feat.loc[reg_train_mask, "finish_position_classified"],
        )

        pred = np.clip(pipe.predict(X_all.loc[reg_val_mask]), 1, 24)

        mae = mean_absolute_error(
            df_feat.loc[reg_val_mask, "finish_position_classified"],
            pred,
        )

        report_rows.append(
            {
                "stage": "regressor_position_if_finished",
                "model": name,
                "validation_year": 2025,
                "rows": int(reg_val_mask.sum()),
                "mae_positions_only_finishers": round(float(mae), 3),
            }
        )

        final_pipe = Pipeline(
            [
                ("prep", make_preprocessor(numeric_features, categorical_features)),
                ("model", model),
            ]
        )

        final_pipe.fit(X_all.loc[finishers_mask], y_pos_finishers)
        fitted_regressors[name] = final_pipe

    all_predictions = []
    all_features = []

    for race in PREDICTION_RACES:
        pred_input = build_prediction_rows(
            df_feat=df_feat,
            circuit_id=race["circuit_id"],
            prediction_round=race["round"],
            race_name=race["race_name"],
        )

        present = set(pred_input["rider_full_name"].tolist())

        missing = [
            name
            for name in REQUESTED_FULL_NAMES
            if name not in present
        ]

        for missing_name in missing:
            report_rows.append(
                {
                    "stage": "prediction_input_warning",
                    "model": "missing_current_2026_row",
                    "race_name": race["race_name"],
                    "note": missing_name,
                }
            )

        for col in categorical_features:
            pred_input[col] = pred_input[col].astype(str)

        # 1. lépcső: célba érési valószínűség.
        for name, pipe in fitted_classifiers.items():
            pred_input[f"prob_finish_{name}"] = pipe.predict_proba(
                pred_input[numeric_features + categorical_features]
            )[:, 1]

        pred_input["prob_finish"] = (
            0.40 * pred_input["prob_finish_RF_cls"]
            + 0.35 * pred_input["prob_finish_ET_cls"]
            + 0.25 * pred_input["prob_finish_GB_cls"]
        )

        pred_input["prob_dnf"] = 1 - pred_input["prob_finish"]

        # 2. lépcső: várható helyezés, ha célba ér.
        for name, pipe in fitted_regressors.items():
            pred_input[f"pred_pos_{name}"] = np.clip(
                pipe.predict(pred_input[numeric_features + categorical_features]),
                1,
                24,
            )

        pred_input["predicted_position_if_finished"] = (
            0.40 * pred_input["pred_pos_RF_reg"]
            + 0.35 * pred_input["pred_pos_ET_reg"]
            + 0.25 * pred_input["pred_pos_GB_reg"]
        )

        fill_cols = [
            "season_classified_avg_before",
            "classified_avg_last5",
            "classified_avg_last3",
            "circuit_classified_avg",
            "mfr_circuit_classified_avg",
        ]

        # Biztonságos NaN-kezelés:
        # ha egy oszlop mediánja is NaN, akkor fallbacket adunk neki.
        for col in fill_cols:
            median_value = pred_input[col].median(skipna=True)

            if pd.isna(median_value):
                if col in ["circuit_classified_avg", "mfr_circuit_classified_avg"]:
                    median_value = pred_input["season_classified_avg_before"].median(skipna=True)

                if pd.isna(median_value):
                    median_value = pred_input["classified_avg_last5"].median(skipna=True)

                if pd.isna(median_value):
                    median_value = pred_input["career_classified_avg_before"].median(skipna=True)

                if pd.isna(median_value):
                    median_value = 15.0

            pred_input[col + "_filled"] = pred_input[col].fillna(median_value)

        pred_input["classified_form_score"] = (
            0.35 * pred_input["season_classified_avg_before_filled"]
            + 0.25 * pred_input["classified_avg_last5_filled"]
            + 0.15 * pred_input["classified_avg_last3_filled"]
            + 0.15 * pred_input["circuit_classified_avg_filled"]
            + 0.10 * pred_input["mfr_circuit_classified_avg_filled"]
        )

        pred_input["pure_pace_score"] = (
            0.60 * pred_input["predicted_position_if_finished"]
            + 0.40 * pred_input["classified_form_score"]
        )

        pred_input["dnf_risk_penalty"] = (
            4.0 * pred_input["prob_dnf"]
            + 1.2 * pred_input["season_dnf_rate_before"].fillna(0)
            + 0.8 * pred_input["dnf_rate_last5"].fillna(0)
        )

        pred_input["expected_position_score"] = (
            pred_input["pure_pace_score"] + pred_input["dnf_risk_penalty"]
        )

        pred_input["risk_level"] = pd.cut(
            pred_input["prob_dnf"],
            bins=[-0.01, 0.25, 0.45, 1.0],
            labels=["alacsony", "közepes", "magas"],
        )

        pred_input = pred_input.sort_values("expected_position_score").reset_index(
            drop=True
        )

        pred_input["predicted_rank"] = np.arange(1, len(pred_input) + 1)

        pred_input["pure_pace_rank"] = pred_input["pure_pace_score"].rank(
            method="first",
            ascending=True,
        )

        pred_input["expected_rank"] = pred_input["expected_position_score"].rank(
            method="first",
            ascending=True,
        )

        all_predictions.append(pred_input.copy())
        all_features.append(pred_input.copy())

    all_pred = pd.concat(all_predictions, ignore_index=True)

    out_cols = [
        "race_name",
        "circuit_id",
        "round",
        "predicted_rank",
        "rider_full_name",
        "bike",
        "team",
        "expected_position_score",
        "pure_pace_score",
        "predicted_position_if_finished",
        "prob_finish",
        "prob_dnf",
        "dnf_risk_penalty",
        "risk_level",
        "season_points_before_calc",
        "champ_pos_before",
        "season_classified_avg_before",
        "classified_avg_last3",
        "classified_avg_last5",
        "classified_avg_last10",
        "circuit_classified_avg",
        "mfr_circuit_classified_avg"
    ]

    all_pred = all_pred[out_cols].copy()

    html = make_prediction_table_html(
        all_pred=all_pred,
        rider_image_urls=RIDER_IMAGE_URLS,
        team_image_urls=TEAM_PROFILE_URLS,
    )
    HTML_OUT = Path("motogp_next5_predictions.html")
    HTML_OUT.write_text(html, encoding="utf-8")
    if HAS_IPYTHON:
        display(HTML(html))
    else:
        print(html)

    print("\nValidáció:")
    print(pd.DataFrame(report_rows).to_string(index=False))


if __name__ == "__main__":
    main()

#,Versenyző képe,Versenyző,Nemzet,Csapat képe,Csapat,Várható score,Tempó score,Ha célba ér,Célba érés,DNF,Kockázat
1,,Manuel González,🇪🇸,,Liqui Moly Dynavolt Intact GP,4.856,4.440,4.451,89.6%,10.4%,alacsony
2,,Senna Agius,🇦🇺,,Liqui Moly Dynavolt Intact GP,6.524,5.753,4.759,80.7%,19.3%,alacsony
3,,Izan Guevara,🇪🇸,,Blu Cru Pramac Yamaha Moto2,7.467,6.648,6.616,79.5%,20.5%,alacsony
4,,David Alonso,🇨🇴,,CFMoto Aspar Team,7.986,6.707,6.545,71.0%,29.0%,közepes
5,,Iván Ortolá,🇪🇸,,QJMotor – MSi,8.075,6.196,5.563,67.0%,33.0%,közepes
6,,Celestino Vietti,🇮🇹,,SpeedRS Team,8.524,6.745,6.901,62.5%,37.5%,közepes
7,,Daniel Holgado,🇪🇸,,CFMoto Aspar Team,8.628,7.242,6.962,68.3%,31.7%,közepes
8,,Filip Salac,🇨🇿,,OnlyFans American Racing Team,8.897,7.788,7.197,72.3%,27.7%,közepes
9,,Collin Veijer,🇳🇱,,Red Bull KTM Ajo,12.107,10.463,9.342,71.9%,28.1%,közepes
10,,Tony Arbolino,🇮🇹,,Reds Fantic Racing,12.333,11.494,11.093,79.0%,21.0%,alacsony



Validáció:
                         stage  model  validation_year  rows  accuracy  brier_loss  roc_auc  mae_positions_only_finishers
 classifier_finish_probability RF_cls             2025   615     0.777       0.179    0.522                           NaN
 classifier_finish_probability ET_cls             2025   615     0.704       0.202    0.508                           NaN
 classifier_finish_probability GB_cls             2025   615     0.831       0.139    0.555                           NaN
regressor_position_if_finished RF_reg             2025   514       NaN         NaN      NaN                         4.369
regressor_position_if_finished ET_reg             2025   514       NaN         NaN      NaN                         4.391
regressor_position_if_finished GB_reg             2025   514       NaN         NaN      NaN                         4.245


# Moto3

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, brier_score_loss, mean_absolute_error, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from IPython.display import HTML, display

from html import escape

try:
    from IPython.display import HTML, display
    HAS_IPYTHON = True
except ImportError:
    HAS_IPYTHON = False

warnings.filterwarnings("ignore")

INPUT = Path("moto3_dataset_from_Race_plus_2026_current.csv")


PREDICTION_RACES = [
    {"round": 10, "circuit_id": "GER", "race_name": "German GP / Sachsenring"},
    {"round": 11, "circuit_id": "GBR", "race_name": "British GP / Silverstone"},
    {"round": 12, "circuit_id": "ARA", "race_name": "Aragon GP"},
    {"round": 13, "circuit_id": "RSM", "race_name": "San Marino GP / Misano"},
    {"round": 14, "circuit_id": "AUT", "race_name": "Austrian GP / Red Bull Ring"},
]
PREDICTION_YEAR = 2026
PREDICTION_ROUND = 10

REQUESTED_FULL_NAMES = [
    "Máximo Quiles",
    "Álvaro Carpe",
    "David Almansa",
    "Brian Uriarte",
    "Marco Morelli",
    "Hakim Danish",
    "Veda Pratama",
    "Valentín Perrone",
    "David Muñoz",
    "Guido Pini",
    "Joel Esteban",
    "Adrián Cruces",
    "Matteo Bertelle",
    "Rico Salmela",
    "Casey O'Gorman",
    "Joel Kelso",
    "Eddie O'Shea",
    "Jesús Ríos",
    "Adrián Fernández",
    "Scott Ogden",
    "Ryusei Yamanaka",
    #"Marcos Uriarte",
    "Leo Rammerstorfer",
    "Cormac Buchanan",
    "Zen Mitani",
    "Ruché Moodley",
    "Nicola Carraro",
]

RIDER_IMAGE_URLS = {
    "Máximo Quiles": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/53b582c3-f578-4fe8-994d-9083bcf92974/HVT7FVm1.png?width=1800",
    "Álvaro Carpe": "https://resources.motogp.pulselive.com/photo-resources/2026/03/13/c37a19b3-0bd0-4629-8f8d-9dc0df2c9b9e/gJg5KXUh.png?width=1800",
    "David Almansa": "https://resources.motogp.pulselive.com/photo-resources/2026/04/22/5883dfc6-4352-42f2-ae88-63588adcb86f/R9qwDoNr.png?width=1800",
    "Brian Uriarte": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/1570bcfd-abbd-4da0-a693-67421d37925b/9NvAFfi1.png?width=1800",
    "Marco Morelli": "https://resources.motogp.pulselive.com/photo-resources/2026/06/02/dca2c8ca-278f-4c59-80c2-39b6b78388e4/GhG4KUTK.png?width=1800",
    "Hakim Danish": "https://resources.motogp.pulselive.com/photo-resources/2026/03/28/d2e35d7b-b224-4fd4-9420-c4a427575390/13-M3-Hakim-Danish-Rider-Official__DSC4649.png?width=1800",
    "Veda Pratama": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/be88dd1e-d632-47b1-a7c2-4d9c889032ac/lD5xewFa.png?width=1800",
    "Valentín Perrone": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/75ff69a7-392f-44ed-a651-dbccf630d8b9/eE62f3F3.png?width=1800",
    "David Muñoz": "https://resources.motogp.pulselive.com/photo-resources/2026/04/22/525450d9-4bf8-4ef5-88ef-16c343a81762/4qQFNAtQ.png?width=1800",
    "Guido Pini": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/ff8a46b9-2660-43f1-a9b0-0ada88901497/rsxvOlc3.png?width=1800",
    "Joel Esteban": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/8edf6171-ab3a-4b83-ba04-d9c7d7201a52/J4mKrZoA.png?width=1800",
    "Adrián Cruces": "https://resources.motogp.pulselive.com/photo-resources/2026/04/22/238c0bdc-7293-47bd-91a5-b28b49505b02/tlgX2dVW.png?width=1800",
    "Matteo Bertelle": "https://resources.motogp.pulselive.com/photo-resources/2026/04/22/5100b377-8044-4e20-9783-eeac12fef8c0/6KFT8DDE.png?width=1800",
    "Rico Salmela": "https://resources.motogp.pulselive.com/photo-resources/2026/04/22/753a8ef5-5cd4-47ce-9772-5fae9fd5c278/nCu1oPHR.png?width=1800",
    "Casey O'Gorman": "https://resources.motogp.pulselive.com/photo-resources/2026/04/22/55a76a34-c14b-4ddd-9230-c5e065a63966/6Gx9Pwmf.png?width=1800",
    "Joel Kelso": "https://resources.motogp.pulselive.com/photo-resources/2026/06/02/c5342049-3dab-4e55-bdb4-9e242314159a/aLVqrZJl.png?width=1800",
    "Eddie O'Shea": "https://resources.motogp.pulselive.com/photo-resources/2026/06/02/dbab1cb3-71fc-419d-9ee7-a195057d7649/pDAqKbSx.png?width=1800",
    "Jesús Ríos": "https://resources.motogp.pulselive.com/photo-resources/2026/04/22/c8737c49-5473-4851-95bf-92e814348db3/za4cubZS.png?width=1800",
    "Adrián Fernández": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/f6fc68bc-d53f-494b-a2f5-b9181eed47d4/9r79Hlqp.png?width=1800",
    "Scott Ogden": "https://resources.motogp.pulselive.com/photo-resources/2026/04/22/8dda3e7b-0ab3-42c5-91e5-307c80e049db/FA4UZd3w.png?width=1800",
    "Ryusei Yamanaka": "https://resources.motogp.pulselive.com/photo-resources/2026/03/28/abaec172-09b4-40dd-b6dc-f95fe882d001/06-M3-Ryusei-Yamanaka-Rider-Official__DSC4607.png?width=1800",
    #"Marcos Uriarte": "https://resources.motogp.pulselive.com/photo-resources/2026/03/28/f203b076-9b04-4ff8-a324-079e8fcf98bc/89-M3-Marcos-Uriarte-M3-_DSC3383.png?width=1800",
    "Leo Rammerstorfer": "https://resources.motogp.pulselive.com/photo-resources/2026/04/22/250368ce-2bd6-4057-916f-9865a4d2c2c0/m1dvfEKX.png?width=1800",
    "Cormac Buchanan": "https://resources.motogp.pulselive.com/photo-resources/2026/04/22/16126702-d2b6-4118-b52c-961ba7b5c694/3LZ1yUa5.png?width=1800",
    "Zen Mitani": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/e9e625be-8f00-4c1f-96ed-8309673fa40c/Bul50N4d.png?width=1800",
    "Ruché Moodley": "https://resources.motogp.pulselive.com/photo-resources/2026/04/22/0df9f811-1c6e-4255-81d0-aef8bbf7804d/mRKoN4hY.png?width=1800",
    "Nicola Carraro": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/19a4fca8-fd31-4662-ad37-4a78a811dc0c/UPU3mwzu.png?width=1800",
}

RIDER_COUNTRY_FLAGS = {
    "Máximo Quiles": "🇪🇸",
    "Álvaro Carpe": "🇪🇸",
    "David Almansa": "🇪🇸",
    "Brian Uriarte": "🇪🇸",
    "Marco Morelli": "🇦🇷",
    "Hakim Danish": "🇲🇾",
    "Veda Pratama": "🇮🇩",
    "Valentín Perrone": "🇦🇷",
    "David Muñoz": "🇪🇸",
    "Guido Pini": "🇮🇹",
    "Joel Esteban": "🇪🇸",
    "Adrián Cruces": "🇪🇸",
    "Matteo Bertelle": "🇮🇹",
    "Rico Salmela": "🇫🇮",
    "Casey O'Gorman": "🇮🇪",
    "Joel Kelso": "🇦🇺",
    "Eddie O'Shea": "🇬🇧",
    "Jesús Ríos": "🇪🇸",
    "Adrián Fernández": "🇪🇸",
    "Scott Ogden": "🇬🇧",
    "Ryusei Yamanaka": "🇯🇵",
    "Marcos Uriarte": "🇪🇸",
    "Leo Rammerstorfer": "🇦🇹",
    "Cormac Buchanan": "🇦🇺",
    "Zen Mitani": "🇯🇵",
    "Ruché Moodley": "🇿🇦",
    "Nicola Carraro": "🇮🇹",
}

TEAM_PROFILE_URLS = {
    "CFMoto Aspar Team": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/c59927fe-e611-4594-9f11-301299bd5d0f/28-M3-Maximo-Quiles-Bike-Official-x01-LG9_0240.png?width=1248",
    "Red Bull KTM Ajo": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/62fa1fe6-e562-4e14-80d3-d5e51a97b7dc/51-M3-Brian-Uriarte-Bike-Official-x01-LG9_0095.png?width=1248",
    "Liqui Moly Dynavolt Intact GP": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/d92f5954-8d9d-41c5-b898-31727ea71466/22-M3-David-Almansa-Bike-Official-x01-LG9_9432.png?width=1248",
    "Aeon Credit – MT Helmets – MSi": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/16300445-f9c8-47b3-b8b4-1dfa7e76a880/06-M3-Ryusei-Yamanaka-Bike-Official-x01-LG9_9975.png?width=1248",
    "Honda Team Asia": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/24b72ea8-c4b8-4e56-9f09-744e1ff4a116/09-M3-Veda-Pratama-Bike-Official-x01-LG9_0213.png?width=1248",
    "Red Bull KTM Tech3": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/1a5c9ed2-be30-4426-a0c2-eab64f138b9d/27-M3-Rico-Salmela-Bike-Official-x01-LG9_9880.png?width=1248",
    "Leopard Racing": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/895c5e1a-48fc-44d0-81fd-0509e36e700a/31-M3-Adrian-Fernandez-Bike-Official-x01-LG9_9500.png?width=1248",
    "LevelUp – MTA": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/3d6d1bd3-1633-4132-b12d-819b155bd93d/18-M3-Matteo-Bertelle-Bike-Official-x01-LG9_0023.png?width=1248",
    "CIP Green Power": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/39a1ebff-d280-433f-afb0-dc37091838c6/11-M3-Adrian-Cruces-Bike-Official-x01-LG9_9619.png?width=1248",
    "Sic58 Squadra Corse": "https://resources.motogp.pulselive.com/photo-resources/2026/02/23/269b5596-fa70-42f8-9af8-391fd1604ae3/05-M3-Leo-Rammerstorfer-Bike-Official-x01-LG9_0460.png?width=1248",
    "Gryd – MLav Racing": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/6e71a100-77a6-4f1e-abd6-ac27f477137c/08-M3-Eddie-O-Shea-Bike-Official-x01-LG9_9665.png?width=1248",
    "Rivacold Snipers Team": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/c6494635-6a67-4eec-b801-a45d3305e3c6/10-M3-Nicola-Carraro-Bike-Official-x01-LG9_9782.png?width=1248",
    "Code Motorsports": "https://resources.motogp.pulselive.com/photo-resources/2026/02/24/f6dccef7-5fd1-4afd-ad38-2f99672c3ac8/21-M3-Ruche-Moodley-Bike-Official-x01-LG9_9528.png?width=1248",
}

def img_tag(url: str, alt: str, width: int = 70) -> str:
    if not isinstance(url, str) or not url.startswith("http"):
        return ""

    return (
        f'<img src="{escape(url)}" '
        f'alt="{escape(alt)}" '
        f'width="{width}" '
        f'style="border-radius:10px; object-fit:cover;">'
    )


def make_prediction_table_html(
    all_pred: pd.DataFrame,
    rider_image_urls: dict,
    team_image_urls: dict,
) -> str:
    css = """
    <style>
        .motogp-wrap {
            font-family: Arial, sans-serif;
            max-width: 1500px;
        }

        .race-title {
            margin-top: 34px;
            margin-bottom: 10px;
            font-size: 24px;
            font-weight: 800;
        }

        table.motogp-table {
            border-collapse: collapse;
            width: 100%;
            margin-bottom: 28px;
            font-size: 14px;
        }

        table.motogp-table th {
            background: #111;
            color: white;
            padding: 9px;
            text-align: left;
        }

        table.motogp-table td {
            border-bottom: 1px solid #ddd;
            padding: 8px;
            vertical-align: middle;
        }

        .rank {
            font-size: 20px;
            font-weight: 800;
            text-align: center;
        }

        .rider-name {
            font-weight: 700;
            font-size: 15px;
        }

        .country {
            font-size: 24px;
            text-align: center;
        }

        .risk-alacsony {
            color: #087f23;
            font-weight: 700;
        }

        .risk-közepes {
            color: #b26a00;
            font-weight: 700;
        }

        .risk-magas {
            color: #b00020;
            font-weight: 700;
        }
    </style>
    """

    html = [css, '<div class="motogp-wrap">']

    for race_name, part in all_pred.groupby("race_name", sort=False):
        part = part.sort_values("predicted_rank").copy()

        html.append(f'<div class="race-title">{escape(str(race_name))}</div>')

        html.append(
            """
            <table class="motogp-table">
                <thead>
                    <tr>
                        <th>#</th>
                        <th>Versenyző képe</th>
                        <th>Versenyző</th>
                        <th>Nemzet</th>
                        <th>Csapat képe</th>
                        <th>Csapat</th>
                        <th>Várható score</th>
                        <th>Tempó score</th>
                        <th>Ha célba ér</th>
                        <th>Célba érés</th>
                        <th>DNF</th>
                        <th>Kockázat</th>
                    </tr>
                </thead>
                <tbody>
            """
        )

        for _, row in part.iterrows():
            rider = str(row["rider_full_name"])
            team = str(row["team"])

            rider_img = img_tag(
                rider_image_urls.get(rider, ""),
                rider,
                width=130,
            )

            team_img = img_tag(
                team_image_urls.get(team, ""),
                team,
                width=110,
            )

            country = RIDER_COUNTRY_FLAGS.get(rider, "")

            risk = str(row["risk_level"])
            risk_class = f"risk-{risk}"

            html.append(
                f"""
                <tr>
                    <td class="rank">{int(row["predicted_rank"])}</td>

                    <td>{rider_img}</td>

                    <td>
                        <div class="rider-name">{escape(rider)}</div>
                    </td>

                    <td class="country">{escape(country)}</td>

                    <td>{team_img}</td>

                    <td>{escape(team)}</td>

                    <td>{row["expected_position_score"]:.3f}</td>

                    <td>{row["pure_pace_score"]:.3f}</td>

                    <td>{row["predicted_position_if_finished"]:.3f}</td>

                    <td>{row["prob_finish"] * 100:.1f}%</td>

                    <td>{row["prob_dnf"] * 100:.1f}%</td>

                    <td class="{risk_class}">{escape(risk)}</td>
                </tr>
                """
            )

        html.append("</tbody></table>")

    html.append("</div>")

    return "\n".join(html)

def safe_one_hot_encoder() -> OneHotEncoder:
    """scikit-learn 1.2+ uses sparse_output, older versions use sparse."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def add_two_stage_features(data: pd.DataFrame) -> pd.DataFrame:
    """
    Feature engineering leakage nélkül.

    Fontos:
    - Ret / DNF / EX / NC nem 25. helyként megy bele a tempóátlagokba.
    - Ezek külön DNF / non-classified kockázatként szerepelnek.
    """
    data = data.sort_values(["year", "round", "circuit_id", "rider_id"]).copy()

    data["classified"] = (data["dnf"].fillna(0).astype(int) == 0).astype(int)
    data["finish_position_classified"] = np.where(
        data["classified"].eq(1),
        data["final_position"].astype(float),
        np.nan,
    )

    # Csak célba ért versenyek tempója, kiesések nélkül.
    for window in [3, 5, 10]:
        data[f"classified_avg_last{window}"] = data.groupby("rider_id")[
            "finish_position_classified"
        ].transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())

        data[f"dnf_rate_last{window}"] = data.groupby("rider_id")["dnf"].transform(
            lambda s: s.shift(1).rolling(window, min_periods=1).mean()
        )

        data[f"top10_rate_last{window}"] = data.groupby("rider_id")[
            "finish_position_classified"
        ].transform(lambda s: (s.shift(1) <= 10).rolling(window, min_periods=1).mean())

    # Szezonon belüli forma az adott futam előtt.
    data["season_classified_avg_before"] = data.groupby(["year", "rider_id"])[
        "finish_position_classified"
    ].transform(lambda s: s.shift(1).expanding().mean())

    data["season_dnf_rate_before"] = data.groupby(["year", "rider_id"])["dnf"].transform(
        lambda s: s.shift(1).expanding().mean()
    )

    data["season_points_before_calc"] = data.groupby(["year", "rider_id"])[
        "points"
    ].transform(lambda s: s.shift(1).fillna(0).cumsum())

    data["season_starts_before"] = data.groupby(["year", "rider_id"]).cumcount()

    # Teljes korábbi karrierforma az adott futam előtt.
    data["career_classified_avg_before"] = data.groupby("rider_id")[
        "finish_position_classified"
    ].transform(lambda s: s.shift(1).expanding().mean())

    data["career_dnf_rate_before"] = data.groupby("rider_id")["dnf"].transform(
        lambda s: s.shift(1).expanding().mean()
    )

    return data


def make_preprocessor(numeric_features, categorical_features):
    return ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), numeric_features),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", safe_one_hot_encoder()),
                    ]
                ),
                categorical_features,
            ),
        ],
        remainder="drop",
    )


def build_prediction_rows(df_feat: pd.DataFrame, circuit_id: str, prediction_round: int, race_name: str) -> pd.DataFrame:
    current = df_feat[df_feat["year"].eq(PREDICTION_YEAR)].copy()

    latest_by_rider = (
        current.sort_values(["round", "circuit_id"])
        .groupby("rider_full_name", as_index=False)
        .tail(1)
        .copy()
    )

    latest_by_rider = latest_by_rider[
        latest_by_rider["rider_full_name"].isin(REQUESTED_FULL_NAMES)
    ].copy()

    rows = []

    for _, last in latest_by_rider.iterrows():
        rider_id = last["rider_id"]
        constructor = last["bike"]

        rider_hist = df_feat[df_feat["rider_id"].eq(rider_id)].sort_values(
            ["year", "round"]
        )

        season_hist = rider_hist[rider_hist["year"].eq(PREDICTION_YEAR)]
        classified_hist = rider_hist["finish_position_classified"].dropna()
        dnf_hist = rider_hist["dnf"].astype(float)

        rider_circuit_hist = df_feat[
            df_feat["circuit_id"].eq(circuit_id)
            & df_feat["rider_id"].eq(rider_id)
        ]

        mfr_circuit_hist = df_feat[
            df_feat["circuit_id"].eq(circuit_id)
            & df_feat["bike"].eq(constructor)
        ]

        rider_circuit_classified = rider_circuit_hist[
            "finish_position_classified"
        ].dropna()

        mfr_circuit_classified = mfr_circuit_hist[
            "finish_position_classified"
        ].dropna()

        rows.append(
            {
                "year": PREDICTION_YEAR,
                "round": prediction_round,
                "circuit_id": circuit_id,
                "race_name": race_name,
                "rider_id": rider_id,
                "rider_name": last["rider_name"],
                "rider_full_name": last["rider_full_name"],
                "team": last["team"],
                "bike": constructor,
                "moto3_experience_years": last.get("moto3_experience_years", np.nan),

                "classified_avg_last3": classified_hist.tail(3).mean(),
                "classified_avg_last5": classified_hist.tail(5).mean(),
                "classified_avg_last10": classified_hist.tail(10).mean(),
                "season_classified_avg_before": season_hist[
                    "finish_position_classified"
                ].mean(),
                "career_classified_avg_before": classified_hist.mean(),
                "circuit_classified_avg": rider_circuit_classified.mean(),
                "mfr_circuit_classified_avg": mfr_circuit_classified.mean(),

                "dnf_rate_last3": dnf_hist.tail(3).mean(),
                "dnf_rate_last5": dnf_hist.tail(5).mean(),
                "dnf_rate_last10": dnf_hist.tail(10).mean(),
                "season_dnf_rate_before": season_hist["dnf"].mean(),
                "career_dnf_rate_before": dnf_hist.mean(),

                "top10_rate_last5": (classified_hist.tail(5) <= 10).mean(),
                "top10_rate_last10": (classified_hist.tail(10) <= 10).mean(),
                "season_points_before_calc": season_hist["points"].sum(),
                "season_starts_before": len(season_hist),
            }
        )

    pred_input = pd.DataFrame(rows)

    pred_input["champ_pos_before"] = pred_input[
        "season_points_before_calc"
    ].rank(method="min", ascending=False)

    return pred_input

def main():
    df = pd.read_csv(INPUT)

    df = df.sort_values(["year", "round", "circuit_id", "rider_id"]).reset_index(
        drop=True
    )

    df_feat = add_two_stage_features(df)

    numeric_features = [
        "round",
        "classified_avg_last3",
        "classified_avg_last5",
        "classified_avg_last10",
        "dnf_rate_last3",
        "dnf_rate_last5",
        "dnf_rate_last10",
        "top10_rate_last5",
        "top10_rate_last10",
        "season_classified_avg_before",
        "season_dnf_rate_before",
        "season_points_before_calc",
        "season_starts_before",
        "career_classified_avg_before",
        "career_dnf_rate_before",
        "moto3_experience_years",
        "champ_pos_before",
    ]

    # Tanítási adatokhoz pálya- és gyártó-pálya statok.
    # Shift(1), hogy az adott futam eredménye ne szivárogjon be feature-ként.
    df_feat["circuit_classified_avg"] = df_feat.groupby(["circuit_id", "rider_id"])[
        "finish_position_classified"
    ].transform(lambda s: s.shift(1).expanding().mean())

    df_feat["mfr_circuit_classified_avg"] = df_feat.groupby(
        ["circuit_id", "bike"]
    )["finish_position_classified"].transform(lambda s: s.shift(1).expanding().mean())

    numeric_features += [
        "circuit_classified_avg",
        "mfr_circuit_classified_avg",
    ]

    categorical_features = [
        "circuit_id",
        "rider_id",
        "bike",
        "team",
    ]

    preprocessor = make_preprocessor(numeric_features, categorical_features)

    classifier_models = {
        "RF_cls": RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=4,
            class_weight="balanced_subsample",
            random_state=101,
            n_jobs=-1,
        ),
        "ET_cls": ExtraTreesClassifier(
            n_estimators=500,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=102,
            n_jobs=-1,
        ),
        "GB_cls": GradientBoostingClassifier(
            n_estimators=180,
            learning_rate=0.04,
            max_depth=3,
            random_state=103,
        ),
    }

    regressor_models = {
        "RF_reg": RandomForestRegressor(
            n_estimators=500,
            min_samples_leaf=4,
            random_state=201,
            n_jobs=-1,
        ),
        "ET_reg": ExtraTreesRegressor(
            n_estimators=500,
            min_samples_leaf=3,
            random_state=202,
            n_jobs=-1,
        ),
        "GB_reg": GradientBoostingRegressor(
            n_estimators=240,
            learning_rate=0.035,
            max_depth=3,
            subsample=0.85,
            random_state=203,
        ),
    }

    X_all = df_feat[numeric_features + categorical_features]
    y_finish = df_feat["classified"].astype(int)

    finishers_mask = df_feat["classified"].eq(1)

    y_pos_finishers = df_feat.loc[
        finishers_mask,
        "finish_position_classified",
    ].astype(float)

    report_rows = []

    # Validáció: 2022–2024 tanulás, 2025 teszt.
    train_mask = df_feat["year"].le(2024)
    val_mask = df_feat["year"].eq(2025)

    fitted_classifiers = {}

    for name, model in classifier_models.items():
        pipe = Pipeline(
            [
                ("prep", make_preprocessor(numeric_features, categorical_features)),
                ("model", model),
            ]
        )

        pipe.fit(X_all.loc[train_mask], y_finish.loc[train_mask])

        proba = pipe.predict_proba(X_all.loc[val_mask])[:, 1]
        pred_label = (proba >= 0.5).astype(int)
        y_true = y_finish.loc[val_mask]

        row = {
            "stage": "classifier_finish_probability",
            "model": name,
            "validation_year": 2025,
            "rows": int(val_mask.sum()),
            "accuracy": round(float(accuracy_score(y_true, pred_label)), 3),
            "brier_loss": round(float(brier_score_loss(y_true, proba)), 3),
        }

        try:
            row["roc_auc"] = round(float(roc_auc_score(y_true, proba)), 3)
        except ValueError:
            row["roc_auc"] = np.nan

        report_rows.append(row)

        final_pipe = Pipeline(
            [
                ("prep", preprocessor),
                ("model", model),
            ]
        )

        final_pipe.fit(X_all, y_finish)
        fitted_classifiers[name] = final_pipe

    fitted_regressors = {}

    reg_train_mask = train_mask & finishers_mask
    reg_val_mask = val_mask & finishers_mask

    for name, model in regressor_models.items():
        pipe = Pipeline(
            [
                ("prep", make_preprocessor(numeric_features, categorical_features)),
                ("model", model),
            ]
        )

        pipe.fit(
            X_all.loc[reg_train_mask],
            df_feat.loc[reg_train_mask, "finish_position_classified"],
        )

        pred = np.clip(pipe.predict(X_all.loc[reg_val_mask]), 1, 24)

        mae = mean_absolute_error(
            df_feat.loc[reg_val_mask, "finish_position_classified"],
            pred,
        )

        report_rows.append(
            {
                "stage": "regressor_position_if_finished",
                "model": name,
                "validation_year": 2025,
                "rows": int(reg_val_mask.sum()),
                "mae_positions_only_finishers": round(float(mae), 3),
            }
        )

        final_pipe = Pipeline(
            [
                ("prep", make_preprocessor(numeric_features, categorical_features)),
                ("model", model),
            ]
        )

        final_pipe.fit(X_all.loc[finishers_mask], y_pos_finishers)
        fitted_regressors[name] = final_pipe

    all_predictions = []
    all_features = []

    for race in PREDICTION_RACES:
        pred_input = build_prediction_rows(
            df_feat=df_feat,
            circuit_id=race["circuit_id"],
            prediction_round=race["round"],
            race_name=race["race_name"],
        )

        present = set(pred_input["rider_full_name"].tolist())

        missing = [
            name
            for name in REQUESTED_FULL_NAMES
            if name not in present
        ]

        for missing_name in missing:
            report_rows.append(
                {
                    "stage": "prediction_input_warning",
                    "model": "missing_current_2026_row",
                    "race_name": race["race_name"],
                    "note": missing_name,
                }
            )

        for col in categorical_features:
            pred_input[col] = pred_input[col].astype(str)

        # 1. lépcső: célba érési valószínűség.
        for name, pipe in fitted_classifiers.items():
            pred_input[f"prob_finish_{name}"] = pipe.predict_proba(
                pred_input[numeric_features + categorical_features]
            )[:, 1]

        pred_input["prob_finish"] = (
            0.40 * pred_input["prob_finish_RF_cls"]
            + 0.35 * pred_input["prob_finish_ET_cls"]
            + 0.25 * pred_input["prob_finish_GB_cls"]
        )

        pred_input["prob_dnf"] = 1 - pred_input["prob_finish"]

        # 2. lépcső: várható helyezés, ha célba ér.
        for name, pipe in fitted_regressors.items():
            pred_input[f"pred_pos_{name}"] = np.clip(
                pipe.predict(pred_input[numeric_features + categorical_features]),
                1,
                24,
            )

        pred_input["predicted_position_if_finished"] = (
            0.40 * pred_input["pred_pos_RF_reg"]
            + 0.35 * pred_input["pred_pos_ET_reg"]
            + 0.25 * pred_input["pred_pos_GB_reg"]
        )

        fill_cols = [
            "season_classified_avg_before",
            "classified_avg_last5",
            "classified_avg_last3",
            "circuit_classified_avg",
            "mfr_circuit_classified_avg",
        ]

        # Biztonságos NaN-kezelés:
        # ha egy oszlop mediánja is NaN, akkor fallbacket adunk neki.
        for col in fill_cols:
            median_value = pred_input[col].median(skipna=True)

            if pd.isna(median_value):
                if col in ["circuit_classified_avg", "mfr_circuit_classified_avg"]:
                    median_value = pred_input["season_classified_avg_before"].median(skipna=True)

                if pd.isna(median_value):
                    median_value = pred_input["classified_avg_last5"].median(skipna=True)

                if pd.isna(median_value):
                    median_value = pred_input["career_classified_avg_before"].median(skipna=True)

                if pd.isna(median_value):
                    median_value = 15.0

            pred_input[col + "_filled"] = pred_input[col].fillna(median_value)

        pred_input["classified_form_score"] = (
            0.35 * pred_input["season_classified_avg_before_filled"]
            + 0.25 * pred_input["classified_avg_last5_filled"]
            + 0.15 * pred_input["classified_avg_last3_filled"]
            + 0.15 * pred_input["circuit_classified_avg_filled"]
            + 0.10 * pred_input["mfr_circuit_classified_avg_filled"]
        )

        pred_input["pure_pace_score"] = (
            0.60 * pred_input["predicted_position_if_finished"]
            + 0.40 * pred_input["classified_form_score"]
        )

        pred_input["dnf_risk_penalty"] = (
            4.0 * pred_input["prob_dnf"]
            + 1.2 * pred_input["season_dnf_rate_before"].fillna(0)
            + 0.8 * pred_input["dnf_rate_last5"].fillna(0)
        )

        pred_input["expected_position_score"] = (
            pred_input["pure_pace_score"] + pred_input["dnf_risk_penalty"]
        )

        pred_input["risk_level"] = pd.cut(
            pred_input["prob_dnf"],
            bins=[-0.01, 0.25, 0.45, 1.0],
            labels=["alacsony", "közepes", "magas"],
        )

        pred_input = pred_input.sort_values("expected_position_score").reset_index(
            drop=True
        )

        pred_input["predicted_rank"] = np.arange(1, len(pred_input) + 1)

        pred_input["pure_pace_rank"] = pred_input["pure_pace_score"].rank(
            method="first",
            ascending=True,
        )

        pred_input["expected_rank"] = pred_input["expected_position_score"].rank(
            method="first",
            ascending=True,
        )

        all_predictions.append(pred_input.copy())
        all_features.append(pred_input.copy())

    all_pred = pd.concat(all_predictions, ignore_index=True)

    out_cols = [
        "race_name",
        "circuit_id",
        "round",
        "predicted_rank",
        "rider_full_name",
        "bike",
        "team",
        "expected_position_score",
        "pure_pace_score",
        "predicted_position_if_finished",
        "prob_finish",
        "prob_dnf",
        "dnf_risk_penalty",
        "risk_level",
        "season_points_before_calc",
        "champ_pos_before",
        "season_classified_avg_before",
        "classified_avg_last3",
        "classified_avg_last5",
        "classified_avg_last10",
        "circuit_classified_avg",
        "mfr_circuit_classified_avg"
    ]

    all_pred = all_pred[out_cols].copy()

    html = make_prediction_table_html(
        all_pred=all_pred,
        rider_image_urls=RIDER_IMAGE_URLS,
        team_image_urls=TEAM_PROFILE_URLS,
    )
    HTML_OUT = Path("motogp_next5_predictions.html")
    HTML_OUT.write_text(html, encoding="utf-8")
    if HAS_IPYTHON:
        display(HTML(html))
    else:
        print(html)

    print("\nValidáció:")
    print(pd.DataFrame(report_rows).to_string(index=False))


if __name__ == "__main__":
    main()

#,Versenyző képe,Versenyző,Nemzet,Csapat képe,Csapat,Várható score,Tempó score,Ha célba ér,Célba érés,DNF,Kockázat
1,,Máximo Quiles,🇪🇸,,CFMoto Aspar Team,3.825,3.516,3.662,92.3%,7.7%,alacsony
2,,Álvaro Carpe,🇪🇸,,Red Bull KTM Ajo,5.838,4.538,4.609,77.5%,22.5%,alacsony
3,,Marco Morelli,🇦🇷,,CFMoto Aspar Team,7.903,7.256,6.938,83.8%,16.2%,alacsony
4,,Brian Uriarte,🇪🇸,,Red Bull KTM Ajo,7.926,6.628,6.430,74.6%,25.4%,közepes
5,,David Almansa,🇪🇸,,Liqui Moly Dynavolt Intact GP,8.114,6.515,6.353,70.7%,29.3%,közepes
6,,Hakim Danish,🇲🇾,,Aeon Credit – MT Helmets – MSi,8.416,7.162,6.944,75.6%,24.4%,alacsony
7,,David Muñoz,🇪🇸,,Liqui Moly Dynavolt Intact GP,9.219,7.487,7.317,65.0%,35.0%,közepes
8,,Veda Pratama,🇮🇩,,Honda Team Asia,10.468,8.502,8.598,60.8%,39.2%,közepes
9,,Valentín Perrone,🇦🇷,,Red Bull KTM Tech3,10.713,9.383,9.067,73.7%,26.3%,közepes
10,,Joel Esteban,🇪🇸,,LevelUp – MTA,11.905,10.109,9.978,68.1%,31.9%,közepes



Validáció:
                         stage  model  validation_year  rows  accuracy  brier_loss  roc_auc  mae_positions_only_finishers
 classifier_finish_probability RF_cls             2025   560     0.787       0.174    0.536                           NaN
 classifier_finish_probability ET_cls             2025   560     0.743       0.192    0.547                           NaN
 classifier_finish_probability GB_cls             2025   560     0.805       0.157    0.563                           NaN
regressor_position_if_finished RF_reg             2025   454       NaN         NaN      NaN                         3.601
regressor_position_if_finished ET_reg             2025   454       NaN         NaN      NaN                         3.753
regressor_position_if_finished GB_reg             2025   454       NaN         NaN      NaN                         3.553


## Exporting results to PNG

In [ ]:
!pip install playwright
!playwright install chromium
!playwright install-deps chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 27.3 MB/s eta 0:00:00
177 MiB [] 0% 359.8s177 MiB [] 0% 17.5s177 MiB [] 0% 14.9s177 MiB [] 0% 15.8s177 MiB [] 0% 7.9s177 MiB [] 1% 5.2s177 MiB [] 2% 3.5s177 MiB [] 3% 3.1s177 MiB [] 4% 3.0s177 MiB [] 5% 2.8s177 MiB [] 5% 2.7s177 MiB [] 6% 2.7s177 MiB [] 6% 3.0s177 MiB [] 7% 3.1s177 MiB [] 7% 3.2s177 MiB [] 7% 3.3s177 MiB [] 8% 3.0s177 MiB [] 9% 2.8s177 MiB [] 10% 2.7s177 MiB [] 11% 2.6s177 MiB [] 12% 2.5s177 MiB [] 13% 2.3s177 MiB [] 14% 2.3s177 MiB [] 15% 2.2s177 MiB [] 16% 2.1s177 MiB [] 18% 2.0s177 MiB [] 19% 1.9s177 MiB [] 20% 1.9s177 MiB [] 21% 1.8s177 MiB [] 22% 1.7s177 MiB [] 23% 1.7s177 MiB [] 25% 1.6s177 MiB [] 25% 1.7s177 MiB [] 27% 1.6s177 MiB [] 28% 1.6s177 MiB [] 30% 1.5s177 MiB [] 32% 1.4s177 MiB [] 33% 1.3s177 MiB [] 35% 1.3s177 MiB [] 36% 1.2s177 MiB [] 38% 1.2s177 MiB [] 39% 1.1s177 MiB [] 41% 1.1s177 MiB [] 42% 1.1s177 MiB [] 44% 1.0s177 MiB [] 44% 1.1s177 MiB [] 45% 1.0s177 MiB [] 46% 1.0s177 MiB [] 48% 1.0s177

In [ ]:
from pathlib import Path
import re
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

HTML_IN = Path("motogp_next5_predictions.html")
EXPORT_DIR = Path("motogp_next5_png")
EXPORT_DIR.mkdir(exist_ok=True)


def slugify_filename(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


html_text = HTML_IN.read_text(encoding="utf-8")
soup = BeautifulSoup(html_text, "html.parser")

# A teljes HTML-ből kimentjük a style blokk(oka)t,
# hogy a külön exportált versenyoldalak is ugyanúgy nézzenek ki.
style_blocks = "\n".join(str(tag) for tag in soup.find_all("style"))

race_sections = []

for title_div in soup.select("div.race-title"):
    race_name = title_div.get_text(strip=True)

    table = title_div.find_next_sibling("table")
    if table is None:
        continue

    section_html = f"""
    <html>
        <head>
            <meta charset="utf-8">
            {style_blocks}
        </head>
        <body>
            <div class="motogp-wrap">
                {str(title_div)}
                {str(table)}
            </div>
        </body>
    </html>
    """

    race_sections.append((race_name, section_html))

print(f"Talált versenyblokkok: {len(race_sections)}")
for race_name, _ in race_sections:
    print("-", race_name)


async with async_playwright() as p:
    browser = await p.chromium.launch(
        headless=True,
        args=[
            "--no-sandbox",
            "--disable-setuid-sandbox",
            "--disable-dev-shm-usage",
            "--disable-gpu",
        ],
    )

    for race_name, section_html in race_sections:
        safe_name = slugify_filename(race_name)

        html_out = EXPORT_DIR / f"{safe_name}.html"
        png_out = EXPORT_DIR / f"{safe_name}.png"

        # opcionálisan külön HTML-ként is elmentjük
        html_out.write_text(section_html, encoding="utf-8")

        page = await browser.new_page(
            viewport={
                "width": 1700,
                "height": 1800,
            },
            device_scale_factor=1,
        )

        await page.set_content(section_html, wait_until="domcontentloaded")

        # kis várakozás a külső képek betöltésére
        await page.wait_for_timeout(5000)

        await page.screenshot(
            path=str(png_out),
            full_page=True,
        )

        await page.close()

        print(f"Kész: {png_out.resolve()}")

    await browser.close()

Talált versenyblokkok: 1
- German GP / Sachsenring
Kész: /content/motogp_next5_png/german_gp_sachsenring.png
